In [1]:
from google.colab import files

uploaded = files.upload()

Saving merged_features_final.xlsx to merged_features_final.xlsx


In [2]:
import pandas as pd

df = pd.read_excel("merged_features_final.xlsx")

print("Shape:", df.shape)
df.head()

Shape: (60000, 20)


,url,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,random_domain,num_dots,num_hyphens,ratio_digits_host,prefix_suffix,num_question_mark,num_eq,domain_age,dns_resolvable,num_ips,ttl,num_name_servers,label
0,https://manua.ls,3.500000,16,0,0.000000,0,8,0,1,0,0.0,0,0,0,-1.0,1.0,1.0,3600.0,4.0,legitimate
1,https://t.co/r77VHQx9lP,4.023137,23,0,0.130435,0,4,1,1,0,0.0,0,0,0,5745.0,1.0,1.0,300.0,8.0,phishing
2,https://sites.google.com/view/attyahoo-plus-la...,4.145908,58,0,0.000000,1,16,0,2,3,0.0,0,0,0,10351.0,1.0,6.0,300.0,4.0,phishing
3,https://bit.ly/48c5385,3.845351,22,0,0.272727,0,6,0,1,0,0.0,0,0,0,6453.0,1.0,2.0,300.0,8.0,phishing
4,https://strato.de,3.381580,17,0,0.000000,0,9,0,1,0,0.0,0,0,0,-1.0,1.0,1.0,3600.0,4.0,legitimate


In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Shape: (60000, 20)

Columns:
['url', 'entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'random_domain', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'prefix_suffix', 'num_question_mark', 'num_eq', 'domain_age', 'dns_resolvable', 'num_ips', 'ttl', 'num_name_servers', 'label']

Data types:
url                   object
entropy              float64
url_length             int64
num_www                int64
ratio_digits_url     float64
num_subdomains         int64
length_hostname        int64
random_domain          int64
num_dots               int64
num_hyphens            int64
ratio_digits_host    float64
prefix_suffix          int64
num_question_mark      int64
num_eq                 int64
domain_age           float64
dns_resolvable       float64
num_ips              float64
ttl                  float64
num_name_servers     float64
label                 object
dtype: object

Missing values:
url                  0
entropy              0
url_leng

In [4]:
print(df["label"].value_counts())
print()
print(df["label"].value_counts(normalize=True) * 100)

label
phishing      31641
legitimate    28359
Name: count, dtype: int64

label
phishing      52.735
legitimate    47.265
Name: proportion, dtype: float64


In [5]:
print("Duplicate URLs:", df["url"].duplicated().sum())

Duplicate URLs: 0


In [6]:
duplicate_urls = df[df["url"].duplicated(keep=False)].sort_values("url")

duplicate_urls.head(20)

,url,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,random_domain,num_dots,num_hyphens,ratio_digits_host,prefix_suffix,num_question_mark,num_eq,domain_age,dns_resolvable,num_ips,ttl,num_name_servers,label


In [7]:
from urllib.parse import urlparse

def extract_domain(url):
    try:
        return urlparse(str(url)).netloc.lower().replace("www.", "")
    except:
        return ""

df["domain"] = df["url"].apply(extract_domain)

df[["url", "domain", "label"]].head(10)

,url,domain,label
0,https://manua.ls,manua.ls,legitimate
1,https://t.co/r77VHQx9lP,t.co,phishing
2,https://sites.google.com/view/attyahoo-plus-la...,sites.google.com,phishing
3,https://bit.ly/48c5385,bit.ly,phishing
4,https://strato.de,strato.de,legitimate
5,https://techmeme.com,techmeme.com,legitimate
6,https://mahabhumi.gov.in,mahabhumi.gov.in,legitimate
7,https://goldenfrog.com,goldenfrog.com,legitimate
8,https://myproductglobalsource.weebly.com/,myproductglobalsource.weebly.com,phishing
9,https://www.dropbox.com/scl/fi/av83rq75mbe8otu...,dropbox.com,phishing


In [8]:
# 1. Number of unique domains
print("Total URLs:", len(df))
print("Unique URLs:", df["url"].nunique())
print("Unique domains:", df["domain"].nunique())

# 2. Repeated domains
domain_counts = df["domain"].value_counts()

print("\nDomains appearing more than once:",
      (domain_counts > 1).sum())

# 3. Domains having both labels
domain_label_counts = df.groupby("domain")["label"].nunique()

conflicting_domains = domain_label_counts[
    domain_label_counts > 1
]

print("\nDomains with BOTH phishing and legitimate URLs:",
      len(conflicting_domains))

Total URLs: 60000
Unique URLs: 60000
Unique domains: 40906

Domains appearing more than once: 3975

Domains with BOTH phishing and legitimate URLs: 52


In [9]:
conflicting_domains = (
    df.groupby("domain")["label"]
    .nunique()
)

conflicting_domains = conflicting_domains[
    conflicting_domains > 1
].index

print("Number of conflicting domains:", len(conflicting_domains))

display(
    df[df["domain"].isin(conflicting_domains)]
    [["url", "domain", "label"]]
    .sort_values("domain")
    .head(100)
)

Number of conflicting domains: 52


,url,domain,label
27791,https://www.art.com,art.com,legitimate
49714,https://www.art.com/,art.com,phishing
37937,https://artisteer.com,artisteer.com,legitimate
38375,http://www.artisteer.com/?p=affr&amp;redirect_...,artisteer.com,phishing
19891,http://awin1.com/cread.php?awinmid=10178&awina...,awin1.com,phishing
...,...,...,...
33397,https://bit.ly/3C1bkYn,bit.ly,phishing
33332,https://bit.ly/482Yepp,bit.ly,phishing
33327,https://bit.ly/4fCcyYB,bit.ly,phishing
33255,https://bit.ly/46xGP7A,bit.ly,phishing


In [10]:
from urllib.parse import urlparse, urlunparse

def normalize_url(url):
    try:
        parsed = urlparse(str(url).strip().lower())

        # Remove trailing slash from path
        path = parsed.path.rstrip("/")

        # Rebuild URL
        normalized = urlunparse((
            parsed.scheme,
            parsed.netloc,
            path,
            "",
            parsed.query,
            ""
        ))

        return normalized

    except:
        return str(url).strip().lower()


df["normalized_url"] = df["url"].apply(normalize_url)

print("Original unique URLs:",
      df["url"].nunique())

print("Normalized unique URLs:",
      df["normalized_url"].nunique())

print("New duplicate URLs after normalization:",
      df["normalized_url"].duplicated().sum())

Original unique URLs: 60000
Normalized unique URLs: 59816
New duplicate URLs after normalization: 184


In [11]:
# Check whether normalized duplicate URLs have conflicting labels

normalized_label_counts = (
    df.groupby("normalized_url")["label"]
    .nunique()
)

conflicting_normalized_urls = normalized_label_counts[
    normalized_label_counts > 1
]

print(
    "Normalized URL groups with conflicting labels:",
    len(conflicting_normalized_urls)
)

Normalized URL groups with conflicting labels: 2


In [12]:
# Show the conflicting cases

conflicting_data = df[
    df["normalized_url"].isin(conflicting_normalized_urls.index)
][["url", "normalized_url", "domain", "label"]].sort_values(
    "normalized_url"
)

display(conflicting_data)

,url,normalized_url,domain,label
5595,https://kakaku.com,https://kakaku.com,kakaku.com,legitimate
20437,https://kakaku.com/,https://kakaku.com,kakaku.com,phishing
27791,https://www.art.com,https://www.art.com,art.com,legitimate
49714,https://www.art.com/,https://www.art.com,art.com,phishing


In [13]:
print("Conflicting normalized URL groups:",
      len(conflicting_normalized_urls))

print("Rows involved in conflicts:",
      df["normalized_url"].isin(
          conflicting_normalized_urls.index
      ).sum())

Conflicting normalized URL groups: 2
Rows involved in conflicts: 4


In [14]:
df_clean = df[
    ~df["normalized_url"].isin(conflicting_normalized_urls.index)
].copy()

print("Original rows:", len(df))
print("After removing conflicting URLs:", len(df_clean))

Original rows: 60000
After removing conflicting URLs: 59996


In [15]:
before = len(df_clean)

df_clean = df_clean.drop_duplicates(
    subset="normalized_url",
    keep="first"
).copy()

print("Rows before:", before)
print("Rows after:", len(df_clean))
print("Rows removed:", before - len(df_clean))

Rows before: 59996
Rows after: 59814
Rows removed: 182


In [16]:
print("Final rows:", len(df_clean))

print(
    "Duplicate normalized URLs:",
    df_clean["normalized_url"].duplicated().sum()
)

print(
    "Conflicting normalized URLs:",
    df_clean.groupby("normalized_url")["label"].nunique()
        .gt(1).sum()
)

Final rows: 59814
Duplicate normalized URLs: 0
Conflicting normalized URLs: 0


In [17]:
print("Shape:", df_clean.shape)

print("\nColumns:")
print(df_clean.columns.tolist())

print("\nData types:")
print(df_clean.dtypes)

Shape: (59814, 22)

Columns:
['url', 'entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'random_domain', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'prefix_suffix', 'num_question_mark', 'num_eq', 'domain_age', 'dns_resolvable', 'num_ips', 'ttl', 'num_name_servers', 'label', 'domain', 'normalized_url']

Data types:
url                   object
entropy              float64
url_length             int64
num_www                int64
ratio_digits_url     float64
num_subdomains         int64
length_hostname        int64
random_domain          int64
num_dots               int64
num_hyphens            int64
ratio_digits_host    float64
prefix_suffix          int64
num_question_mark      int64
num_eq                 int64
domain_age           float64
dns_resolvable       float64
num_ips              float64
ttl                  float64
num_name_servers     float64
label                 object
domain                object
normalized_url        object

In [18]:
missing = df_clean.isnull().sum()

print(missing)
print("\nTotal missing values:", missing.sum())

url                  0
entropy              0
url_length           0
num_www              0
ratio_digits_url     0
num_subdomains       0
length_hostname      0
random_domain        0
num_dots             0
num_hyphens          0
ratio_digits_host    0
prefix_suffix        0
num_question_mark    0
num_eq               0
domain_age           2
dns_resolvable       2
num_ips              2
ttl                  2
num_name_servers     2
label                0
domain               0
normalized_url       0
dtype: int64

Total missing values: 10


In [19]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
entropy,59814.0,4.071604,0.563374,2.353338,3.719295,3.93626,4.221085,6.032729
url_length,59814.0,44.096666,117.899608,12.000000,20.000000,24.00000,41.000000,25523.000000
num_www,59814.0,0.097736,0.298868,0.000000,0.000000,0.00000,0.000000,3.000000
ratio_digits_url,59814.0,0.045274,0.084474,0.000000,0.000000,0.00000,0.067211,0.826087
num_subdomains,59814.0,0.481242,0.575883,0.000000,0.000000,0.00000,1.000000,7.000000
length_hostname,59814.0,16.649965,10.366546,4.000000,10.000000,15.00000,20.000000,100.000000
random_domain,59814.0,0.148360,0.355459,0.000000,0.000000,0.00000,0.000000,1.000000
num_dots,59814.0,1.668339,0.974183,1.000000,1.000000,2.00000,2.000000,23.000000
num_hyphens,59814.0,0.472699,1.100419,0.000000,0.000000,0.00000,1.000000,50.000000
ratio_digits_host,59814.0,0.033040,0.103424,0.000000,0.000000,0.00000,0.000000,0.800000


In [20]:
# Check sentinel values and missing values in DNS/domain features

dns_cols = [
    "domain_age",
    "dns_resolvable",
    "num_ips",
    "ttl",
    "num_name_servers"
]

for col in dns_cols:
    print(f"\n===== {col} =====")
    print("Missing:", df_clean[col].isna().sum())
    print("Value counts:")
    print(df_clean[col].value_counts(dropna=False).head(10))


===== domain_age =====
Missing: 2
Value counts:
domain_age
-1.0        18611
 10351.0     3119
 10352.0     1198
 7234.0      1116
 6453.0       993
 4841.0       788
 5099.0       567
 14306.0      507
 7235.0       457
 4842.0       447
Name: count, dtype: int64

===== dns_resolvable =====
Missing: 2
Value counts:
dns_resolvable
1.0    55675
0.0     4137
NaN        2
Name: count, dtype: int64

===== num_ips =====
Missing: 2
Value counts:
num_ips
1.0     21624
2.0     18442
4.0      6884
6.0      4996
0.0      4137
3.0      3242
5.0       247
8.0       146
7.0        27
10.0       26
Name: count, dtype: int64

===== ttl =====
Missing: 2
Value counts:
ttl
 300.0      25520
 60.0        9794
-1.0         4137
 3600.0      4117
 21600.0     3235
 900.0       3208
 600.0       2597
 20.0        1840
 14400.0     1064
 30.0         589
Name: count, dtype: int64

===== num_name_servers =====
Missing: 2
Value counts:
num_name_servers
4.0     25414
2.0     20445
0.0      4294
3.0      3299
6

In [21]:
print("URLs > 200:", (df_clean["url_length"] > 200).sum())
print("URLs > 500:", (df_clean["url_length"] > 500).sum())
print("URLs > 1000:", (df_clean["url_length"] > 1000).sum())

display(
    df_clean.nlargest(10, "url_length")[
        ["url", "url_length", "label"]
    ]
)

URLs > 200: 583
URLs > 500: 74
URLs > 1000: 21


,url,url_length,label
28140,https://authjgxszcloxpnassocl.firebaseapp.com/...,25523,phishing
2773,https://sobakemozhno.ru/wp-content/plugins/wor...,1520,phishing
7647,https://u24148341.ct.sendgrid.net/ls/click?upn...,1436,phishing
35311,https://sub.followerswave.com/track-followers/...,1261,phishing
49799,http://copia.vspabogados.com/wp-admin/app.php?...,1223,phishing
6134,https://mytimemakeup.com/l/6nmDqdoAkutDBlOyhZb...,1201,phishing
54267,https://toystodolls.com/l/TFXxpwdh0V0rcHBvw8Tk...,1201,phishing
29761,http://fieldconnect.jbktechnologies.net/core/v...,1186,phishing
49826,https://recaptcha-metacloud.com/two_step_verif...,1185,phishing
46647,https://forcarorbike.com/l/Z0MuClXHjtoBCnxPQiP...,1154,phishing


In [22]:
url_features = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

feature_means = (
    df_clean.groupby("label")[url_features]
    .mean()
    .T
)

display(feature_means)

label,legitimate,phishing
entropy,3.699512,4.407027
url_length,20.826639,65.073497
num_www,0.160772,0.040913
ratio_digits_url,0.005438,0.081184
num_subdomains,0.160348,0.770512
length_hostname,12.826639,20.096513
random_domain,0.099482,0.192421
num_dots,1.237261,2.056935
num_hyphens,0.055683,0.848619
ratio_digits_host,0.009543,0.054223


In [23]:
domain_label = (
    df_clean.groupby("domain")["label"]
    .agg(["count", "nunique"])
)

print("Total domains:", len(domain_label))

print("\nDomains containing only one label:")
print(
    (domain_label["nunique"] == 1).sum()
)

print("\nDomains containing both labels:")
print(
    (domain_label["nunique"] > 1).sum()
)

Total domains: 40904

Domains containing only one label:
40854

Domains containing both labels:
50


In [24]:
url_features = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

X = df_clean[url_features]
y = df_clean["label"]

groups = df_clean["domain"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of groups:", groups.nunique())

X shape: (59814, 13)
y shape: (59814,)
Number of groups: 40904


In [25]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)

Classes: ['legitimate' 'phishing']


In [26]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y_encoded, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y_encoded[train_idx]
y_test = y_encoded[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 47323
Testing rows: 12491


In [27]:
train_domains = set(groups_train)
test_domains = set(groups_test)

overlap = train_domains.intersection(test_domains)

print("Training domains:", len(train_domains))
print("Testing domains:", len(test_domains))
print("Overlapping domains:", len(overlap))

Training domains: 32723
Testing domains: 8181
Overlapping domains: 0


In [28]:
import pandas as pd

print("Training distribution:")
print(
    pd.Series(y_train)
    .map({0: "legitimate", 1: "phishing"})
    .value_counts(normalize=True) * 100
)

print("\nTesting distribution:")
print(
    pd.Series(y_test)
    .map({0: "legitimate", 1: "phishing"})
    .value_counts(normalize=True) * 100
)

Training distribution:
phishing      52.088836
legitimate    47.911164
Name: proportion, dtype: float64

Testing distribution:
phishing      54.495237
legitimate    45.504763
Name: proportion, dtype: float64


In [29]:
print(X_train.shape)
print(X_test.shape)

print("\nTraining missing values:")
print(X_train.isnull().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum())

(47323, 13)
(12491, 13)

Training missing values:
entropy              0
url_length           0
num_www              0
ratio_digits_url     0
num_subdomains       0
length_hostname      0
random_domain        0
num_dots             0
num_hyphens          0
ratio_digits_host    0
prefix_suffix        0
num_question_mark    0
num_eq               0
dtype: int64

Testing missing values:
entropy              0
url_length           0
num_www              0
ratio_digits_url     0
num_subdomains       0
length_hostname      0
random_domain        0
num_dots             0
num_hyphens          0
ratio_digits_host    0
prefix_suffix        0
num_question_mark    0
num_eq               0
dtype: int64


In [30]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [31]:
# Predict classes
y_pred_lr = logistic_model.predict(X_test)

# Probability that each URL is phishing
y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions completed!")

Predictions completed!


In [32]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall   :", recall_score(y_test, y_pred_lr))
print("F1 Score :", f1_score(y_test, y_pred_lr))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_lr))

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=label_encoder.classes_
    )
)

Accuracy : 0.9916740052838043
Precision: 0.9985125687936933
Recall   : 0.9861906860584693
F1 Score : 0.9923133776792313
ROC-AUC  : 0.9948789108202665

Classification Report:

              precision    recall  f1-score   support

  legitimate       0.98      1.00      0.99      5684
    phishing       1.00      0.99      0.99      6807

    accuracy                           0.99     12491
   macro avg       0.99      0.99      0.99     12491
weighted avg       0.99      0.99      0.99     12491



In [33]:
import pandas as pd

coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": logistic_model.coef_[0]
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

display(coefficients)

,feature,coefficient,abs_coefficient
2,num_www,-5.944282,5.944282
4,num_subdomains,5.568151,5.568151
1,url_length,4.063343,4.063343
5,length_hostname,-3.805339,3.805339
9,ratio_digits_host,2.718853,2.718853
3,ratio_digits_url,2.219695,2.219695
8,num_hyphens,1.192515,1.192515
0,entropy,-0.799002,0.799002
6,random_domain,-0.603861,0.603861
7,num_dots,-0.177126,0.177126


In [34]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [35]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [36]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_rf))

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred_rf,
    target_names=label_encoder.classes_
))

Accuracy : 0.991273717076295
Precision: 0.9970317601662214
Recall   : 0.9869252240340826
F1 Score : 0.991952750092285
ROC-AUC  : 0.9979784956641582

Classification Report:

              precision    recall  f1-score   support

  legitimate       0.98      1.00      0.99      5684
    phishing       1.00      0.99      0.99      6807

    accuracy                           0.99     12491
   macro avg       0.99      0.99      0.99     12491
weighted avg       0.99      0.99      0.99     12491



In [37]:
rf_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(rf_importance)

,feature,importance
0,entropy,0.256516
1,url_length,0.248009
5,length_hostname,0.138630
3,ratio_digits_url,0.096186
4,num_subdomains,0.075616
2,num_www,0.069469
7,num_dots,0.051071
8,num_hyphens,0.027339
9,ratio_digits_host,0.015851
6,random_domain,0.006612


In [38]:
df_corr = df_clean[url_features].copy()

df_corr["label_encoded"] = (
    df_clean["label"] == "phishing"
).astype(int)

correlations = (
    df_corr.corr()["label_encoded"]
    .drop("label_encoded")
    .sort_values(key=abs, ascending=False)
)

display(correlations)

,label_encoded
entropy,0.627088
num_subdomains,0.529056
ratio_digits_url,0.447735
num_dots,0.420136
num_hyphens,0.359807
length_hostname,0.350173
num_question_mark,0.314538
num_eq,0.244216
ratio_digits_host,0.215717
num_www,-0.200253


In [39]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaled_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

scaled_lr.fit(X_train, y_train)

print("Scaled Logistic Regression trained successfully!")

Scaled Logistic Regression trained successfully!


In [40]:
y_pred_scaled = scaled_lr.predict(X_test)

y_prob_scaled = scaled_lr.predict_proba(X_test)[:, 1]

In [41]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy :", accuracy_score(y_test, y_pred_scaled))
print("Precision:", precision_score(y_test, y_pred_scaled))
print("Recall   :", recall_score(y_test, y_pred_scaled))
print("F1 Score :", f1_score(y_test, y_pred_scaled))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_scaled))

Accuracy : 0.9782243215114883
Precision: 0.9960528313344467
Recall   : 0.9638607315998237
F1 Score : 0.9796923995819024
ROC-AUC  : 0.990240724791003


In [42]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred_scaled,
        target_names=label_encoder.classes_
    )
)

              precision    recall  f1-score   support

  legitimate       0.96      1.00      0.98      5684
    phishing       1.00      0.96      0.98      6807

    accuracy                           0.98     12491
   macro avg       0.98      0.98      0.98     12491
weighted avg       0.98      0.98      0.98     12491



In [43]:
scaled_lr_model = scaled_lr.named_steps["model"]

scaled_coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": scaled_lr_model.coef_[0]
})

scaled_coefficients["abs_coefficient"] = (
    scaled_coefficients["coefficient"].abs()
)

scaled_coefficients = scaled_coefficients.sort_values(
    "abs_coefficient",
    ascending=False
)

display(scaled_coefficients)

,feature,coefficient,abs_coefficient
1,url_length,39.204277,39.204277
3,ratio_digits_url,4.376677,4.376677
5,length_hostname,-4.134753,4.134753
9,ratio_digits_host,-3.485750,3.485750
4,num_subdomains,3.317998,3.317998
8,num_hyphens,3.257121,3.257121
0,entropy,2.902089,2.902089
2,num_www,-1.883480,1.883480
11,num_question_mark,1.125723,1.125723
12,num_eq,0.725048,0.725048


In [44]:
features_no_length = [
    "entropy",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

X_train_no_length = X_train[features_no_length]
X_test_no_length = X_test[features_no_length]

In [45]:
lr_no_length = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr_no_length.fit(
    X_train_no_length,
    y_train
)

LogisticRegression(max_iter=2000, random_state=42)

In [46]:
y_pred_no_length = lr_no_length.predict(X_test_no_length)
y_prob_no_length = lr_no_length.predict_proba(
    X_test_no_length
)[:, 1]

print("Accuracy :",
      accuracy_score(y_test, y_pred_no_length))

print("Precision:",
      precision_score(y_test, y_pred_no_length))

print("Recall   :",
      recall_score(y_test, y_pred_no_length))

print("F1 Score :",
      f1_score(y_test, y_pred_no_length))

print("ROC-AUC  :",
      roc_auc_score(y_test, y_prob_no_length))

Accuracy : 0.962933311984629
Precision: 0.9844227244960293
Recall   : 0.946966358160717
F1 Score : 0.9653313365780607
ROC-AUC  : 0.9820091438347348


In [47]:
import numpy as np

df_clean["log_url_length"] = np.log1p(
    df_clean["url_length"]
)

print(
    df_clean[
        ["url_length", "log_url_length"]
    ].describe()
)

         url_length  log_url_length
count  59814.000000    59814.000000
mean      44.096666        3.495553
std      117.899608        0.661092
min       12.000000        2.564949
25%       20.000000        3.044522
50%       24.000000        3.218876
75%       41.000000        3.737670
max    25523.000000       10.147374


In [48]:
log_features = [
    "entropy",
    "log_url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

In [49]:
X_log = df_clean[log_features]

X_train_log = X_log.iloc[train_idx]
X_test_log = X_log.iloc[test_idx]

In [50]:
lr_log = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr_log.fit(
    X_train_log,
    y_train
)

LogisticRegression(max_iter=2000, random_state=42)

In [51]:
y_pred_log = lr_log.predict(X_test_log)

y_prob_log = lr_log.predict_proba(
    X_test_log
)[:, 1]

print("Accuracy :",
      accuracy_score(y_test, y_pred_log))

print("Precision:",
      precision_score(y_test, y_pred_log))

print("Recall   :",
      recall_score(y_test, y_pred_log))

print("F1 Score :",
      f1_score(y_test, y_pred_log))

print("ROC-AUC  :",
      roc_auc_score(y_test, y_prob_log))

Accuracy : 0.9839084140581218
Precision: 0.9969906710803491
Recall   : 0.9734097252827971
F1 Score : 0.985059094625734
ROC-AUC  : 0.9948094114319336


In [52]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [53]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

XGBoost trained successfully!


In [54]:
y_pred_xgb = xgb_model.predict(X_test)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [55]:
print("Accuracy :",
      accuracy_score(y_test, y_pred_xgb))

print("Precision:",
      precision_score(y_test, y_pred_xgb))

print("Recall   :",
      recall_score(y_test, y_pred_xgb))

print("F1 Score :",
      f1_score(y_test, y_pred_xgb))

print("ROC-AUC  :",
      roc_auc_score(y_test, y_prob_xgb))

Accuracy : 0.9927948122648307
Precision: 0.9988118223674439
Recall   : 0.9879535771999413
F1 Score : 0.9933530280649926
ROC-AUC  : 0.9979972984923517


In [56]:
print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred_xgb,
        target_names=label_encoder.classes_
    )
)


Classification Report:

              precision    recall  f1-score   support

  legitimate       0.99      1.00      0.99      5684
    phishing       1.00      0.99      0.99      6807

    accuracy                           0.99     12491
   macro avg       0.99      0.99      0.99     12491
weighted avg       0.99      0.99      0.99     12491



In [57]:
xgb_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(xgb_importance)

,feature,importance
0,entropy,0.198639
1,url_length,0.171328
2,num_www,0.147780
4,num_subdomains,0.117610
5,length_hostname,0.089346
3,ratio_digits_url,0.073683
9,ratio_digits_host,0.046478
12,num_eq,0.043932
7,num_dots,0.039194
6,random_domain,0.030090


In [58]:
test_results = df_clean.iloc[test_idx].copy()

test_results["actual"] = y_test
test_results["predicted"] = y_pred_xgb
test_results["phishing_probability"] = y_prob_xgb

In [59]:
false_negatives = test_results[
    (test_results["actual"] == 1) &
    (test_results["predicted"] == 0)
].copy()

print("False negatives:", len(false_negatives))

display(
    false_negatives[
        [
            "url",
            "entropy",
            "url_length",
            "num_www",
            "num_subdomains",
            "length_hostname",
            "ratio_digits_url",
            "phishing_probability"
        ]
    ].head(20)
)

False negatives: 82


,url,entropy,url_length,num_www,num_subdomains,length_hostname,ratio_digits_url,phishing_probability
104,https://particulierchequeenergie.com/,3.985556,37,0,0,28,0.0,0.263155
234,https://www-nomura.njatx.cn,4.004344,27,1,1,19,0.0,0.156472
1455,http://cw-net.de,3.452820,16,0,0,9,0.0,0.146194
3882,https://fireflyskinz.com/,4.133661,25,0,0,16,0.0,0.440122
4193,https://www.workjunction.in/,3.797903,28,1,1,19,0.0,0.166868
5850,http://synergyminerals.in,3.893661,25,0,0,18,0.0,0.020057
7890,https://reipojtpoerjp.com,3.623465,25,0,0,17,0.0,0.023288
9290,https://internetbankingonline.digital,3.817995,37,0,0,29,0.0,0.144280
9526,https://www.peage-renouvellement.com/,4.080415,37,1,1,28,0.0,0.479991
9834,http://limonizza.de/,3.784184,20,0,0,12,0.0,0.009640


In [60]:
false_positives = test_results[
    (test_results["actual"] == 0) &
    (test_results["predicted"] == 1)
].copy()

print("False positives:", len(false_positives))

display(
    false_positives[
        [
            "url",
            "entropy",
            "url_length",
            "num_www",
            "num_subdomains",
            "length_hostname",
            "ratio_digits_url",
            "phishing_probability"
        ]
    ].head(20)
)

False positives: 8


,url,entropy,url_length,num_www,num_subdomains,length_hostname,ratio_digits_url,phishing_probability
7106,https://moushikomi-uketsukekun.com,3.822182,34,0,0,26,0.00000,0.515733
24997,https://0123456789nonexistent.com,4.415458,33,0,0,25,0.30303,0.547047
36293,https://marketingautomation.services,4.041336,36,0,0,28,0.00000,0.505879
37923,https://uspreventiveservicestaskforce.org,3.945984,41,0,0,33,0.00000,0.894661
40874,https://www.uspreventiveservicestaskforce.org,4.055649,45,1,1,37,0.00000,0.817843
41147,https://www.kingsoft-office-service.com,4.156195,39,1,1,31,0.00000,0.917707
53906,https://washclothexcludingincluding.com,4.162981,39,0,0,31,0.00000,0.665978
55790,https://use-application-dns.net,3.913401,31,0,0,23,0.00000,0.579338


In [61]:
test_results["uncertainty"] = abs(
    test_results["phishing_probability"] - 0.5
)

uncertain_urls = test_results.sort_values(
    "uncertainty"
).head(20)

display(
    uncertain_urls[
        [
            "url",
            "label",
            "phishing_probability",
            "entropy",
            "url_length",
            "num_subdomains",
            "ratio_digits_url"
        ]
    ]
)

,url,label,phishing_probability,entropy,url_length,num_subdomains,ratio_digits_url
59676,https://askgriff.com/,phishing,0.499261,3.880180,21,0,0.0
36293,https://marketingautomation.services,legitimate,0.505879,4.041336,36,0,0.0
18659,https://eximiamedical.ro/,phishing,0.491847,3.943465,25,0,0.0
22365,https://ksatnam.com/,phishing,0.510813,3.546439,20,0,0.0
58305,https://jorimts.com/,phishing,0.510813,3.546439,20,0,0.0
17338,https://jal-co.bond/,phishing,0.486252,3.884184,20,0,0.0
7106,https://moushikomi-uketsukekun.com,legitimate,0.515733,3.822182,34,0,0.0
22038,https://axsroute.com/,phishing,0.516220,3.748995,21,0,0.0
5238,https://www.fortunebusinessinsights.com,legitimate,0.483649,4.009135,39,1,0.0
9526,https://www.peage-renouvellement.com/,phishing,0.479991,4.080415,37,1,0.0


In [62]:
import joblib

joblib.dump(xgb_model, "phishing_xgb_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("Model saved successfully!")

Model saved successfully!


In [63]:
from urllib.parse import urlparse
import math
from collections import Counter
import re


def calculate_entropy(text):
    if not text:
        return 0

    counts = Counter(text)
    length = len(text)

    entropy = 0

    for count in counts.values():
        probability = count / length
        entropy -= probability * math.log2(probability)

    return entropy


def extract_url_features(url):

    # Make sure URL has a scheme
    if not re.match(r"^https?://", url, re.IGNORECASE):
        url = "http://" + url

    parsed = urlparse(url)

    hostname = parsed.hostname or ""
    full_url = url

    # -----------------------------
    # Basic URL features
    # -----------------------------

    url_length = len(full_url)

    num_www = full_url.lower().count("www")

    digits_url = sum(
        char.isdigit() for char in full_url
    )

    ratio_digits_url = (
        digits_url / url_length
        if url_length > 0 else 0
    )

    # -----------------------------
    # Hostname features
    # -----------------------------

    hostname_length = len(hostname)

    digits_host = sum(
        char.isdigit() for char in hostname
    )

    ratio_digits_host = (
        digits_host / hostname_length
        if hostname_length > 0 else 0
    )

    # -----------------------------
    # Subdomains
    # -----------------------------

    hostname_parts = hostname.split(".")

    num_subdomains = max(
        len(hostname_parts) - 2,
        0
    )

    # -----------------------------
    # URL characters
    # -----------------------------

    num_dots = full_url.count(".")

    num_hyphens = full_url.count("-")

    num_question_mark = full_url.count("?")

    num_eq = full_url.count("=")

    # -----------------------------
    # Domain features
    # -----------------------------

    domain = hostname

    prefix_suffix = 1 if "-" in domain else 0

    # -----------------------------
    # Entropy
    # -----------------------------

    entropy = calculate_entropy(full_url)

    # -----------------------------
    # Return features
    # -----------------------------

    features = {
        "entropy": entropy,
        "url_length": url_length,
        "num_www": num_www,
        "ratio_digits_url": ratio_digits_url,
        "num_subdomains": num_subdomains,
        "length_hostname": hostname_length,
        "random_domain": 0,  # we'll handle this separately
        "num_dots": num_dots,
        "num_hyphens": num_hyphens,
        "ratio_digits_host": ratio_digits_host,
        "prefix_suffix": prefix_suffix,
        "num_question_mark": num_question_mark,
        "num_eq": num_eq
    }

    return features

In [64]:
test_url = "https://www.google.com/search?q=machine+learning"

features = extract_url_features(test_url)

features

{'entropy': 4.334147917027245,
 'url_length': 48,
 'num_www': 1,
 'ratio_digits_url': 0.0,
 'num_subdomains': 1,
 'length_hostname': 14,
 'random_domain': 0,
 'num_dots': 2,
 'num_hyphens': 0,
 'ratio_digits_host': 0.0,
 'prefix_suffix': 0,
 'num_question_mark': 1,
 'num_eq': 1}

In [65]:
sample_url = df_clean.iloc[0]["url"]

print("URL:")
print(sample_url)

print("\nDataset features:")

display(
    df_clean[
        df_clean["url"] == sample_url
    ][[
        "entropy",
        "url_length",
        "num_www",
        "ratio_digits_url",
        "num_subdomains",
        "length_hostname",
        "random_domain",
        "num_dots",
        "num_hyphens",
        "ratio_digits_host",
        "prefix_suffix",
        "num_question_mark",
        "num_eq"
    ]]
)

URL:
https://manua.ls

Dataset features:


,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,random_domain,num_dots,num_hyphens,ratio_digits_host,prefix_suffix,num_question_mark,num_eq
0,3.5,16,0,0.0,0,8,0,1,0,0.0,0,0,0


In [66]:
print("\nOur extracted features:")

display(
    pd.DataFrame([extract_url_features(sample_url)])
)


Our extracted features:


,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,random_domain,num_dots,num_hyphens,ratio_digits_host,prefix_suffix,num_question_mark,num_eq
0,3.5,16,0,0.0,0,8,0,1,0,0.0,0,0,0


In [67]:
import pandas as pd
import numpy as np

feature_columns = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "random_domain",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "prefix_suffix",
    "num_question_mark",
    "num_eq"
]

sample = df_clean.sample(
    n=100,
    random_state=42
)

mismatches = []

for idx, row in sample.iterrows():

    extracted = extract_url_features(row["url"])

    for feature in feature_columns:

        dataset_value = row[feature]
        extracted_value = extracted[feature]

        if not np.isclose(
            dataset_value,
            extracted_value,
            atol=1e-6
        ):
            mismatches.append({
                "index": idx,
                "url": row["url"],
                "feature": feature,
                "dataset": dataset_value,
                "extracted": extracted_value
            })

mismatch_df = pd.DataFrame(mismatches)

print("Total comparisons:",
      len(sample) * len(feature_columns))

print("Total mismatches:",
      len(mismatch_df))

display(mismatch_df.head(20))

Total comparisons: 1300
Total mismatches: 32


,index,url,feature,dataset,extracted
0,28418,https://dc.gov,random_domain,1,0
1,18878,https://mytdatasecureicu.app/v/s/a/session_ind...,random_domain,1,0
2,47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,random_domain,1,0
3,47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,prefix_suffix,0,1
4,30572,https://deliveroo.co.uk,num_subdomains,0,1
5,14692,https://us17.campaign-archive.com/?e=5a66c928e...,random_domain,1,0
6,27632,https://q-r.to/bfTF0j?IQ=74003j3d,random_domain,1,0
7,12232,https://helpdesk-umwedu.weebly.com/,prefix_suffix,0,1
8,19099,https://identifiez-vous.godaddysites.com/,prefix_suffix,0,1
9,30037,https://t.co/fhLWCgBW4J,random_domain,1,0


In [71]:
import tldextract

def get_num_subdomains(url):

    ext = tldextract.extract(url)

    if not ext.subdomain:
        return 0

    return len(ext.subdomain.split("."))

In [69]:
!pip install -q tldextract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 3.3 MB/s eta 0:00:00


In [72]:
test_urls = [
    "https://www.chevrolet.com.ar",
    "https://www.google.com",
    "https://mail.google.com",
    "https://a.b.example.com"
]

for url in test_urls:
    print(url, "→", get_num_subdomains(url))

https://www.chevrolet.com.ar → 1
https://www.google.com → 1
https://mail.google.com → 1
https://a.b.example.com → 2


In [73]:
import tldextract

def get_num_subdomains(url):
    ext = tldextract.extract(url)

    if not ext.subdomain:
        return 0

    return len(ext.subdomain.split("."))

In [74]:
test_urls = [
    "https://www.google.com",
    "https://mail.google.com",
    "https://a.b.example.com",
    "https://www.chevrolet.com.ar",
    "https://example.co.uk"
]

for url in test_urls:
    print(url, "→", get_num_subdomains(url))

https://www.google.com → 1
https://mail.google.com → 1
https://a.b.example.com → 2
https://www.chevrolet.com.ar → 1
https://example.co.uk → 0


In [75]:
prefix_mismatches = mismatch_df[
    mismatch_df["feature"] == "prefix_suffix"
]

display(prefix_mismatches)

,index,url,feature,dataset,extracted
3,47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,prefix_suffix,0,1
7,12232,https://helpdesk-umwedu.weebly.com/,prefix_suffix,0,1
8,19099,https://identifiez-vous.godaddysites.com/,prefix_suffix,0,1
11,13284,https://home-io-trzo-login.pages.dev/,prefix_suffix,0,1
13,57969,https://validate-bell0-mailbox.weebly.com/,prefix_suffix,0,1
15,47404,https://pub-2850cd5d83b9406498615dc0f50afc0d.r...,prefix_suffix,0,1
17,34770,https://help-trzorstart.gitbook.io/us/,prefix_suffix,0,1
22,18378,https://pub-713a6cc1335d4ba1b83057a9bd89b15a.r...,prefix_suffix,0,1
27,52627,https://exod-usweb.pages.dev,prefix_suffix,0,1
29,37585,https://ionos-webmail-de-ionos-id-login-ionos....,prefix_suffix,0,1


In [76]:
prefix_indices = prefix_mismatches["index"].tolist()

display(
    df_clean.loc[
        prefix_indices,
        ["url", "domain", "prefix_suffix"]
    ]
)

,url,domain,prefix_suffix
47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,cv76108-wordpress-zmpmr.tw1.ru,0
12232,https://helpdesk-umwedu.weebly.com/,helpdesk-umwedu.weebly.com,0
19099,https://identifiez-vous.godaddysites.com/,identifiez-vous.godaddysites.com,0
13284,https://home-io-trzo-login.pages.dev/,home-io-trzo-login.pages.dev,0
57969,https://validate-bell0-mailbox.weebly.com/,validate-bell0-mailbox.weebly.com,0
47404,https://pub-2850cd5d83b9406498615dc0f50afc0d.r...,pub-2850cd5d83b9406498615dc0f50afc0d.r2.dev,0
34770,https://help-trzorstart.gitbook.io/us/,help-trzorstart.gitbook.io,0
18378,https://pub-713a6cc1335d4ba1b83057a9bd89b15a.r...,pub-713a6cc1335d4ba1b83057a9bd89b15a.r2.dev,0
52627,https://exod-usweb.pages.dev,exod-usweb.pages.dev,0
37585,https://ionos-webmail-de-ionos-id-login-ionos....,ionos-webmail-de-ionos-id-login-ionos.blogspot...,0


In [77]:
random_mismatches = mismatch_df[
    mismatch_df["feature"] == "random_domain"
]

display(random_mismatches)

,index,url,feature,dataset,extracted
0,28418,https://dc.gov,random_domain,1,0
1,18878,https://mytdatasecureicu.app/v/s/a/session_ind...,random_domain,1,0
2,47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,random_domain,1,0
5,14692,https://us17.campaign-archive.com/?e=5a66c928e...,random_domain,1,0
6,27632,https://q-r.to/bfTF0j?IQ=74003j3d,random_domain,1,0
9,30037,https://t.co/fhLWCgBW4J,random_domain,1,0
10,2359,https://q-r.to/bfMR70,random_domain,1,0
14,47404,https://pub-2850cd5d83b9406498615dc0f50afc0d.r...,random_domain,1,0
16,25466,https://cdn-fc.com,random_domain,1,0
18,30238,https://expedition-relais.com/as.php,random_domain,1,0


In [78]:
random_indices = random_mismatches["index"].tolist()

display(
    df_clean.loc[
        random_indices,
        ["url", "domain", "random_domain"]
    ]
)

,url,domain,random_domain
28418,https://dc.gov,dc.gov,1
18878,https://mytdatasecureicu.app/v/s/a/session_ind...,mytdatasecureicu.app,1
47723,https://cv76108-wordpress-zmpmr.tw1.ru/wp-cont...,cv76108-wordpress-zmpmr.tw1.ru,1
14692,https://us17.campaign-archive.com/?e=5a66c928e...,us17.campaign-archive.com,1
27632,https://q-r.to/bfTF0j?IQ=74003j3d,q-r.to,1
30037,https://t.co/fhLWCgBW4J,t.co,1
2359,https://q-r.to/bfMR70,q-r.to,1
47404,https://pub-2850cd5d83b9406498615dc0f50afc0d.r...,pub-2850cd5d83b9406498615dc0f50afc0d.r2.dev,1
25466,https://cdn-fc.com,cdn-fc.com,1
30238,https://expedition-relais.com/as.php,expedition-relais.com,1


In [79]:
# Check XGBoost importance of the two difficult features

importance_df = pd.DataFrame({
    "feature": url_features,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df)

,feature,importance
0,entropy,0.198639
1,url_length,0.171328
2,num_www,0.147780
4,num_subdomains,0.117610
5,length_hostname,0.089346
3,ratio_digits_url,0.073683
9,ratio_digits_host,0.046478
12,num_eq,0.043932
7,num_dots,0.039194
6,random_domain,0.030090


In [80]:
clean_features = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    # "random_domain",       # removed
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    # "prefix_suffix",       # removed
    "num_question_mark",
    "num_eq"
]

X_train_clean = X_train[clean_features]
X_test_clean = X_test[clean_features]

xgb_clean = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_clean.fit(X_train_clean, y_train)

y_pred_clean = xgb_clean.predict(X_test_clean)
y_prob_clean = xgb_clean.predict_proba(X_test_clean)[:, 1]

In [81]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

clean_results = {
    "Accuracy": accuracy_score(y_test, y_pred_clean),
    "Precision": precision_score(y_test, y_pred_clean),
    "Recall": recall_score(y_test, y_pred_clean),
    "F1": f1_score(y_test, y_pred_clean),
    "ROC-AUC": roc_auc_score(y_test, y_prob_clean)
}

clean_results

{'Accuracy': 0.9931150428308382,
 'Precision': 0.998812527831379,
 'Recall': 0.988541207580432,
 'F1': 0.9936503248670998,
 'ROC-AUC': np.float64(0.9979547304400704)}

In [82]:
url_features_final = [
    "entropy",
    "url_length",
    "num_www",
    "ratio_digits_url",
    "num_subdomains",
    "length_hostname",
    "num_dots",
    "num_hyphens",
    "ratio_digits_host",
    "num_question_mark",
    "num_eq"
]

In [83]:
import math
import re
from collections import Counter
from urllib.parse import urlparse
import tldextract


def calculate_entropy(text):
    if not text:
        return 0

    counter = Counter(text)
    length = len(text)

    entropy = 0

    for count in counter.values():
        probability = count / length
        entropy -= probability * math.log2(probability)

    return entropy


def extract_url_features(url):

    # -----------------------------
    # 1. Parse URL
    # -----------------------------
    parsed = urlparse(url)

    hostname = parsed.hostname or ""


    # -----------------------------
    # 2. Extract registered domain
    #    and subdomains
    # -----------------------------
    ext = tldextract.extract(url)

    if ext.subdomain:
        num_subdomains = len(ext.subdomain.split("."))
    else:
        num_subdomains = 0


    # -----------------------------
    # 3. URL-level features
    # -----------------------------
    url_length = len(url)

    entropy = calculate_entropy(url)

    num_www = url.lower().count("www")

    digit_count_url = sum(char.isdigit() for char in url)

    ratio_digits_url = (
        digit_count_url / url_length
        if url_length > 0
        else 0
    )

    num_dots = url.count(".")

    num_hyphens = url.count("-")

    num_question_mark = url.count("?")

    num_eq = url.count("=")


    # -----------------------------
    # 4. Hostname-level features
    # -----------------------------
    length_hostname = len(hostname)

    digit_count_host = sum(
        char.isdigit() for char in hostname
    )

    ratio_digits_host = (
        digit_count_host / length_hostname
        if length_hostname > 0
        else 0
    )


    # -----------------------------
    # 5. Return exact feature order
    # -----------------------------
    return {
        "entropy": entropy,
        "url_length": url_length,
        "num_www": num_www,
        "ratio_digits_url": ratio_digits_url,
        "num_subdomains": num_subdomains,
        "length_hostname": length_hostname,
        "num_dots": num_dots,
        "num_hyphens": num_hyphens,
        "ratio_digits_host": ratio_digits_host,
        "num_question_mark": num_question_mark,
        "num_eq": num_eq
    }

In [84]:
test_url = "https://manua.ls"

features = extract_url_features(test_url)

features

{'entropy': 3.5,
 'url_length': 16,
 'num_www': 0,
 'ratio_digits_url': 0.0,
 'num_subdomains': 0,
 'length_hostname': 8,
 'num_dots': 1,
 'num_hyphens': 0,
 'ratio_digits_host': 0.0,
 'num_question_mark': 0,
 'num_eq': 0}

In [85]:
import random
import pandas as pd

random.seed(42)

sample_indices = random.sample(
    list(df_clean.index),
    100
)

mismatches = []

for idx in sample_indices:

    url = df_clean.loc[idx, "url"]

    extracted = extract_url_features(url)

    for feature in url_features_final:

        dataset_value = df_clean.loc[idx, feature]
        extracted_value = extracted[feature]

        # Allow tiny floating-point differences
        if isinstance(dataset_value, float):

            if not math.isclose(
                dataset_value,
                extracted_value,
                rel_tol=1e-9,
                abs_tol=1e-9
            ):
                mismatches.append({
                    "index": idx,
                    "url": url,
                    "feature": feature,
                    "dataset": dataset_value,
                    "extracted": extracted_value
                })

        else:

            if dataset_value != extracted_value:
                mismatches.append({
                    "index": idx,
                    "url": url,
                    "feature": feature,
                    "dataset": dataset_value,
                    "extracted": extracted_value
                })


mismatch_df_final = pd.DataFrame(mismatches)

print("Total comparisons:", len(sample_indices) * len(url_features_final))
print("Total mismatches:", len(mismatch_df_final))

display(mismatch_df_final)

Total comparisons: 1100
Total mismatches: 0


""


In [86]:
from xgboost import XGBClassifier

final_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

final_xgb_model.fit(
    X_train[ url_features_final ],
    y_train
)

print("Final XGBoost model trained successfully.")

Final XGBoost model trained successfully.


In [87]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

y_pred_final = final_xgb_model.predict(
    X_test[url_features_final]
)

y_prob_final = final_xgb_model.predict_proba(
    X_test[url_features_final]
)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall   :", recall_score(y_test, y_pred_final))
print("F1 Score :", f1_score(y_test, y_pred_final))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_final))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_final))

Accuracy : 0.9931150428308382
Precision: 0.998812527831379
Recall   : 0.988541207580432
F1 Score : 0.9936503248670998
ROC-AUC  : 0.9979547304400704

Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      5684
           1       1.00      0.99      0.99      6807

    accuracy                           0.99     12491
   macro avg       0.99      0.99      0.99     12491
weighted avg       0.99      0.99      0.99     12491



In [88]:
import joblib

joblib.dump(
    final_xgb_model,
    "phishing_xgb_model.pkl"
)

joblib.dump(
    url_features_final,
    "feature_names.pkl"
)

print("Model saved successfully.")
print("Feature list saved successfully.")

Model saved successfully.
Feature list saved successfully.


In [89]:
def predict_url(url):

    # Extract features
    features = extract_url_features(url)

    # Convert dictionary to DataFrame
    X_input = pd.DataFrame(
        [features],
        columns=url_features_final
    )

    # Get probability
    phishing_probability = final_xgb_model.predict_proba(
        X_input
    )[0][1]

    # Prediction
    if phishing_probability >= 0.5:
        prediction = "Phishing"
    else:
        prediction = "Legitimate"

    return {
        "url": url,
        "prediction": prediction,
        "phishing_probability": phishing_probability,
        "legitimate_probability": 1 - phishing_probability
    }

In [90]:
result = predict_url("https://www.google.com")

result

{'url': 'https://www.google.com',
 'prediction': 'Legitimate',
 'phishing_probability': np.float32(0.005525957),
 'legitimate_probability': np.float32(0.99447405)}

In [91]:
result = predict_url("https://www.github.com")

result

{'url': 'https://www.github.com',
 'prediction': 'Legitimate',
 'phishing_probability': np.float32(0.0046431352),
 'legitimate_probability': np.float32(0.99535686)}

In [92]:
result = predict_url(
    "http://secure-login-account-verification-12345.com/login"
)

result

{'url': 'http://secure-login-account-verification-12345.com/login',
 'prediction': 'Phishing',
 'phishing_probability': np.float32(0.93720645),
 'legitimate_probability': np.float32(0.06279355)}

In [93]:
unseen_test_urls = [
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.amazon.com",
    "https://www.wikipedia.org",
    "https://www.python.org",
    "https://stackoverflow.com",
    "https://www.linkedin.com",
    "https://www.netflix.com",

    "http://secure-account-login-verification-92837.com/login",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "http://bank-account-verification-18472.com/login"
]

In [94]:
unseen_test_urls = [
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.amazon.com",
    "https://www.wikipedia.org",
    "https://www.python.org",
    "https://stackoverflow.com",
    "https://www.linkedin.com",
    "https://www.netflix.com",

    "http://secure-account-login-verification-92837.com/login",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "http://bank-account-verification-18472.com/login"
]

In [95]:
import tldextract

for url in unseen_test_urls:

    ext = tldextract.extract(url)

    if ext.registered_domain:
        domain = ext.registered_domain
    else:
        domain = ext.domain

    domain_exists = (
        df_clean["domain"]
        .astype(str)
        .str.lower()
        .eq(domain.lower())
        .any()
    )

    url_exists = (
        df_clean["normalized_url"]
        .astype(str)
        .str.lower()
        .eq(url.lower().rstrip("/"))
        .any()
    )

    print(
        url,
        "| domain in dataset:",
        domain_exists,
        "| URL in dataset:",
        url_exists
    )

/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = ext.registered_domain
/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_do

https://www.microsoft.com | domain in dataset: True | URL in dataset: False
https://www.apple.com | domain in dataset: False | URL in dataset: False
https://www.amazon.com | domain in dataset: True | URL in dataset: False
https://www.wikipedia.org | domain in dataset: True | URL in dataset: False
https://www.python.org | domain in dataset: True | URL in dataset: False


/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = ext.registered_domain
/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_do

https://stackoverflow.com | domain in dataset: False | URL in dataset: False
https://www.linkedin.com | domain in dataset: False | URL in dataset: False
https://www.netflix.com | domain in dataset: True | URL in dataset: False
http://secure-account-login-verification-92837.com/login | domain in dataset: False | URL in dataset: False
http://paypal-account-security-check-48291.com/verify | domain in dataset: False | URL in dataset: False
http://free-prize-winner-92731.com/claim | domain in dataset: False | URL in dataset: False
http://bank-account-verification-18472.com/login | domain in dataset: False | URL in dataset: False


/tmp/ipykernel_1303/313667722.py:7: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  if ext.registered_domain:
/tmp/ipykernel_1303/313667722.py:8: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = ext.registered_domain


In [96]:
results = []

for url in unseen_test_urls:

    result = predict_url(url)

    results.append({
        "URL": url,
        "Prediction": result["prediction"],
        "Phishing Probability": float(
            result["phishing_probability"]
        )
    })

unseen_results = pd.DataFrame(results)

display(unseen_results)

,URL,Prediction,Phishing Probability
0,https://www.microsoft.com,Legitimate,0.009345
1,https://www.apple.com,Legitimate,0.009769
2,https://www.amazon.com,Legitimate,0.003954
3,https://www.wikipedia.org,Legitimate,0.018236
4,https://www.python.org,Legitimate,0.004724
5,https://stackoverflow.com,Legitimate,0.021552
6,https://www.linkedin.com,Legitimate,0.004804
7,https://www.netflix.com,Legitimate,0.009535
8,http://secure-account-login-verification-92837...,Phishing,0.937206
9,http://paypal-account-security-check-48291.com...,Phishing,0.882909


In [97]:
dns_features = [
    "domain_age",
    "dns_resolvable",
    "num_ips",
    "ttl",
    "num_name_servers"
]

display(
    df_clean[dns_features].describe(include="all").T
)

,count,mean,std,min,25%,50%,75%,max
domain_age,59812.0,4616.968083,4279.347374,-1.0,-1.0,4558.0,8611.0,15148.0
dns_resolvable,59812.0,0.930833,0.253740,0.0,1.0,1.0,1.0,1.0
num_ips,59812.0,2.159099,1.652231,0.0,1.0,2.0,3.0,28.0
ttl,59812.0,2017.700478,5192.450180,-1.0,60.0,300.0,600.0,21600.0
num_name_servers,59812.0,3.267705,1.743243,0.0,2.0,4.0,4.0,22.0


In [98]:
for feature in dns_features:
    print("\n==============================")
    print(feature)
    print("==============================")
    print(df_clean[feature].value_counts(dropna=False).head(20))


domain_age
domain_age
-1.0        18611
 10351.0     3119
 10352.0     1198
 7234.0      1116
 6453.0       993
 4841.0       788
 5099.0       567
 14306.0      507
 7235.0       457
 4842.0       447
 6454.0       375
 5745.0       327
 5100.0       321
 3249.0       274
 6953.0       236
 11161.0      229
 5001.0       226
 9089.0       203
 7373.0       195
 14307.0      172
Name: count, dtype: int64

dns_resolvable
dns_resolvable
1.0    55675
0.0     4137
NaN        2
Name: count, dtype: int64

num_ips
num_ips
1.0     21624
2.0     18442
4.0      6884
6.0      4996
0.0      4137
3.0      3242
5.0       247
8.0       146
7.0        27
10.0       26
9.0        13
11.0        7
12.0        6
16.0        4
18.0        3
20.0        3
NaN         2
14.0        2
15.0        1
27.0        1
Name: count, dtype: int64

ttl
ttl
 300.0      25520
 60.0        9794
-1.0         4137
 3600.0      4117
 21600.0     3235
 900.0       3208
 600.0       2597
 20.0        1840
 14400.0     1064
 

In [99]:
display(
    df_clean.groupby("label")[dns_features].mean()
)

,domain_age,dns_resolvable,num_ips,ttl,num_name_servers
label,,,,,
legitimate,5294.548965,0.993723,1.959128,2011.495786,3.197306
phishing,4006.122079,0.874138,2.339374,2023.294071,3.331172


In [100]:
for feature in dns_features:

    print("\n==============================")
    print(feature)
    print("==============================")

    display(
        df_clean.groupby("label")[feature]
        .describe()
    )


domain_age


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,5294.548965,4175.070058,-1.0,370.0,5397.0,9298.0,15148.0
phishing,31455.0,4006.122079,4280.316907,-1.0,-1.0,3068.0,7234.0,14307.0



dns_resolvable


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,0.993723,0.078981,0.0,1.0,1.0,1.0,1.0
phishing,31455.0,0.874138,0.331699,0.0,1.0,1.0,1.0,1.0



num_ips


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,1.959128,1.265970,0.0,1.0,2.0,2.0,28.0
phishing,31455.0,2.339374,1.917698,0.0,1.0,2.0,4.0,12.0



ttl


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,2011.495786,4908.352303,-1.0,120.0,300.0,600.0,21600.0
phishing,31455.0,2023.294071,5435.927643,-1.0,60.0,300.0,300.0,21600.0



num_name_servers


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
legitimate,28357.0,3.197306,1.507155,0.0,2.0,3.0,4.0,22.0
phishing,31455.0,3.331172,1.929326,0.0,2.0,4.0,4.0,10.0


In [101]:
dns_failed = df_clean[
    df_clean["dns_resolvable"] == 0
]

display(
    dns_failed[
        [
            "url",
            "domain",
            "label",
            "domain_age",
            "dns_resolvable",
            "num_ips",
            "ttl",
            "num_name_servers"
        ]
    ].head(30)
)

,url,domain,label,domain_age,dns_resolvable,num_ips,ttl,num_name_servers
38,https://share.hsforms.com/14mI_206kTRGfwzo7boR...,share.hsforms.com,phishing,-1.0,0.0,0.0,-1.0,0.0
40,https://l.wl.co/l?u=https://qr-codes.io/p8BGXc,l.wl.co,phishing,-1.0,0.0,0.0,-1.0,0.0
50,https://allegrolokalnie.0b3918v.shop/,allegrolokalnie.0b3918v.shop,phishing,-1.0,0.0,0.0,-1.0,0.0
54,https://poczta.ncelodir.cfd/payment/cae44c860d...,poczta.ncelodir.cfd,phishing,-1.0,0.0,0.0,-1.0,0.0
55,http://43.156.237.181/ServiceLogin?passive=120...,43.156.237.181,phishing,-1.0,0.0,0.0,-1.0,0.0
74,http://tekdatagt.com/mail.verify/Sites/index.html,tekdatagt.com,phishing,-1.0,0.0,0.0,-1.0,0.0
81,https://mashreghnews.ir,mashreghnews.ir,legitimate,-1.0,0.0,0.0,-1.0,0.0
84,https://www.mobiletoken-bancasella.com/indexip...,mobiletoken-bancasella.com,phishing,107.0,0.0,0.0,-1.0,0.0
88,https://avend.cfd/jp/,avend.cfd,phishing,-1.0,0.0,0.0,-1.0,0.0
106,http://43.156.0.130/InteractiveLogin?continue=...,43.156.0.130,phishing,-1.0,0.0,0.0,-1.0,0.0


In [102]:
dns_relationship = pd.crosstab(
    df_clean["dns_resolvable"],
    df_clean["num_ips"]
)

display(dns_relationship)

num_ips,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,14.0,15.0,16.0,18.0,20.0,27.0,28.0
dns_resolvable,,,,,,,,,,,,,,,,,,,,
0.0,4137,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1.0,0,21624,18442,3242,6884,247,4996,27,146,13,26,7,6,2,1,4,3,3,1,1


In [103]:
display(
    df_clean[
        ["dns_resolvable", "num_ips", "ttl", "num_name_servers"]
    ]
    .groupby("dns_resolvable")
    .agg(["mean", "min", "max"])
)

num_ips                     ttl                \
                    mean  min   max         mean  min      max   
dns_resolvable                                                   
0.0             0.000000  0.0   0.0    -1.000000 -1.0     -1.0   
1.0             2.319533  1.0  28.0  2167.702524  0.0  21600.0   

               num_name_servers             
                           mean  min   max  
dns_resolvable                              
0.0                    0.000000  0.0   0.0  
1.0                    3.510516  0.0  22.0

In [104]:
domain_age_unknown = df_clean[
    df_clean["domain_age"] == -1
]

display(
    domain_age_unknown[
        [
            "url",
            "domain",
            "label",
            "domain_age",
            "dns_resolvable",
            "num_ips",
            "ttl",
            "num_name_servers"
        ]
    ].head(30)
)

,url,domain,label,domain_age,dns_resolvable,num_ips,ttl,num_name_servers
0,https://manua.ls,manua.ls,legitimate,-1.0,1.0,1.0,3600.0,4.0
4,https://strato.de,strato.de,legitimate,-1.0,1.0,1.0,3600.0,4.0
11,https://interactivedns.com,interactivedns.com,legitimate,-1.0,1.0,1.0,300.0,2.0
15,https://flow.page/vbybtvvrtby65gf4,flow.page,phishing,-1.0,1.0,2.0,300.0,2.0
21,https://locker-prbs-fr.com/,locker-prbs-fr.com,phishing,-1.0,1.0,1.0,900.0,3.0
23,https://urlz.fr/uRb2,urlz.fr,phishing,-1.0,1.0,2.0,300.0,2.0
24,https://pitch.com,pitch.com,legitimate,-1.0,1.0,4.0,60.0,4.0
25,https://culturedcode.com,culturedcode.com,legitimate,-1.0,1.0,4.0,60.0,4.0
28,https://applesupport-2oa.pages.dev/,applesupport-2oa.pages.dev,phishing,-1.0,1.0,2.0,300.0,2.0
29,https://q-r.to/bfeYdf,q-r.to,phishing,-1.0,1.0,4.0,60.0,4.0


In [105]:
print("Total domain_age = -1:",
      len(domain_age_unknown))

print("\nLabel distribution:")
display(
    domain_age_unknown["label"].value_counts()
)

Total domain_age = -1: 18611

Label distribution:


,count
label,
phishing,12052
legitimate,6559


In [106]:
dns_summary = (
    df_clean
    .groupby("label")["dns_resolvable"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("percentage")
    .reset_index()
)

display(dns_summary)

,label,dns_resolvable,percentage
0,legitimate,1.0,99.372289
1,legitimate,0.0,0.627711
2,phishing,1.0,87.413766
3,phishing,0.0,12.586234


In [107]:
dns_features = [
    "domain_age",
    "dns_resolvable",
    "num_ips",
    "ttl",
    "num_name_servers"
]

In [108]:
print("X_train rows:", len(X_train))
print("X_test rows:", len(X_test))

print("First train indices:", X_train.index[:5].tolist())
print("First test indices:", X_test.index[:5].tolist())

X_train rows: 47323
X_test rows: 12491
First train indices: [0, 2, 3, 4, 5]
First test indices: [1, 9, 14, 19, 25]


In [109]:
X_train_dns_raw = df_clean.loc[
    X_train.index,
    dns_features
]

X_test_dns_raw = df_clean.loc[
    X_test.index,
    dns_features
]

print("DNS train shape:", X_train_dns_raw.shape)
print("DNS test shape :", X_test_dns_raw.shape)

DNS train shape: (47323, 5)
DNS test shape : (12491, 5)


In [110]:
from sklearn.impute import SimpleImputer

dns_imputer = SimpleImputer(strategy="median")

X_train_dns = dns_imputer.fit_transform(
    X_train_dns_raw
)

X_test_dns = dns_imputer.transform(
    X_test_dns_raw
)

print("DNS train shape:", X_train_dns.shape)
print("DNS test shape :", X_test_dns.shape)

DNS train shape: (47323, 5)
DNS test shape : (12491, 5)


In [111]:
from xgboost import XGBClassifier

dns_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

dns_model.fit(
    X_train_dns,
    y_train
)

print("DNS model trained successfully.")

DNS model trained successfully.


In [112]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

dns_pred = dns_model.predict(X_test_dns)

dns_prob = dns_model.predict_proba(
    X_test_dns
)[:, 1]

dns_results = {
    "Accuracy": accuracy_score(y_test, dns_pred),
    "Precision": precision_score(y_test, dns_pred),
    "Recall": recall_score(y_test, dns_pred),
    "F1": f1_score(y_test, dns_pred),
    "ROC-AUC": roc_auc_score(y_test, dns_prob)
}

dns_results

{'Accuracy': 0.7058682251220879,
 'Precision': 0.8388492321003678,
 'Recall': 0.5697076538857059,
 'F1': 0.678565179352581,
 'ROC-AUC': np.float64(0.7755068570489851)}

In [113]:
dns_importance = pd.DataFrame({
    "feature": dns_features,
    "importance": dns_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(dns_importance)

,feature,importance
1,dns_resolvable,0.419229
2,num_ips,0.202377
0,domain_age,0.177349
4,num_name_servers,0.110885
3,ttl,0.090160


In [114]:
combined_features = url_features_final + dns_features

print("Total features:", len(combined_features))
print(combined_features)

Total features: 16
['entropy', 'url_length', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq', 'domain_age', 'dns_resolvable', 'num_ips', 'ttl', 'num_name_servers']


In [115]:
X_train_combined_url = df_clean.loc[
    X_train.index,
    url_features_final
]

X_test_combined_url = df_clean.loc[
    X_test.index,
    url_features_final
]

In [116]:
X_train_combined_dns = df_clean.loc[
    X_train.index,
    dns_features
]

X_test_combined_dns = df_clean.loc[
    X_test.index,
    dns_features
]

In [117]:
dns_imputer_combined = SimpleImputer(strategy="median")

X_train_dns_imputed = dns_imputer_combined.fit_transform(
    X_train_combined_dns
)

X_test_dns_imputed = dns_imputer_combined.transform(
    X_test_combined_dns
)

In [118]:
import numpy as np

X_train_combined = np.hstack([
    X_train_combined_url.values,
    X_train_dns_imputed
])

X_test_combined = np.hstack([
    X_test_combined_url.values,
    X_test_dns_imputed
])

print("Combined train shape:", X_train_combined.shape)
print("Combined test shape :", X_test_combined.shape)

Combined train shape: (47323, 16)
Combined test shape : (12491, 16)


In [119]:
combined_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

combined_xgb.fit(
    X_train_combined,
    y_train
)

print("Combined model trained successfully.")

Combined model trained successfully.


In [120]:
combined_pred = combined_xgb.predict(
    X_test_combined
)

combined_prob = combined_xgb.predict_proba(
    X_test_combined
)[:, 1]

combined_results = {
    "Accuracy": accuracy_score(
        y_test,
        combined_pred
    ),
    "Precision": precision_score(
        y_test,
        combined_pred
    ),
    "Recall": recall_score(
        y_test,
        combined_pred
    ),
    "F1": f1_score(
        y_test,
        combined_pred
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        combined_prob
    )
}

combined_results

{'Accuracy': 0.9931150428308382,
 'Precision': 0.9982209043736101,
 'Recall': 0.9891288379609225,
 'F1': 0.9936540731995277,
 'ROC-AUC': np.float64(0.9992782557013018)}

In [121]:
comparison = pd.DataFrame({
    "URL-only": {
        "Accuracy": accuracy_score(y_test, y_pred_final),
        "Precision": precision_score(y_test, y_pred_final),
        "Recall": recall_score(y_test, y_pred_final),
        "F1": f1_score(y_test, y_pred_final),
        "ROC-AUC": roc_auc_score(y_test, y_prob_final)
    },

    "DNS-only": dns_results,

    "URL + DNS": combined_results
})

display(comparison)

,URL-only,DNS-only,URL + DNS
Accuracy,0.993115,0.705868,0.993115
Precision,0.998813,0.838849,0.998221
Recall,0.988541,0.569708,0.989129
F1,0.993650,0.678565,0.993654
ROC-AUC,0.997955,0.775507,0.999278


In [122]:
combined_importance = pd.DataFrame({
    "feature": combined_features,
    "importance": combined_xgb.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(combined_importance)

,feature,importance
0,entropy,0.244756
1,url_length,0.170260
2,num_www,0.115897
5,length_hostname,0.083579
12,dns_resolvable,0.078929
3,ratio_digits_url,0.074248
4,num_subdomains,0.065923
8,ratio_digits_host,0.055738
6,num_dots,0.028688
11,domain_age,0.023088


In [123]:
combined_errors = df_clean.loc[X_test.index].copy()

combined_errors["actual"] = y_test
combined_errors["predicted"] = combined_pred
combined_errors["probability"] = combined_prob

combined_errors["error"] = (
    combined_errors["actual"] !=
    combined_errors["predicted"]
)

display(
    combined_errors[combined_errors["error"]][
        [
            "url",
            "label",
            "domain_age",
            "dns_resolvable",
            "num_ips",
            "ttl",
            "num_name_servers",
            "probability",
            "predicted"
        ]
    ].head(30)
)

,url,label,domain_age,dns_resolvable,num_ips,ttl,num_name_servers,probability,predicted
234,https://www-nomura.njatx.cn,phishing,1078.0,1.0,1.0,600.0,2.0,0.099862,0
1455,http://cw-net.de,phishing,-1.0,1.0,1.0,21600.0,2.0,0.010224,0
3882,https://fireflyskinz.com/,phishing,198.0,1.0,2.0,300.0,2.0,0.194481,0
4193,https://www.workjunction.in/,phishing,2512.0,1.0,2.0,300.0,2.0,0.080414,0
4616,https://hardened-php.net,legitimate,2388.0,1.0,1.0,14400.0,2.0,0.574713,1
5650,https://satisfactory-calculator.com,legitimate,-1.0,1.0,1.0,300.0,2.0,0.585035,1
5850,http://synergyminerals.in,phishing,-1.0,1.0,1.0,14400.0,2.0,0.331251,0
7890,https://reipojtpoerjp.com,phishing,280.0,1.0,2.0,300.0,2.0,0.054150,0
9290,https://internetbankingonline.digital,phishing,-1.0,1.0,2.0,300.0,2.0,0.356833,0
9834,http://limonizza.de/,phishing,-1.0,1.0,1.0,600.0,3.0,0.029887,0


In [124]:
combined_importance = pd.DataFrame({
    "feature": combined_features,
    "importance": combined_xgb.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(combined_importance)

,feature,importance
0,entropy,0.244756
1,url_length,0.170260
2,num_www,0.115897
5,length_hostname,0.083579
12,dns_resolvable,0.078929
3,ratio_digits_url,0.074248
4,num_subdomains,0.065923
8,ratio_digits_host,0.055738
6,num_dots,0.028688
11,domain_age,0.023088


In [125]:
# Predictions from URL-only model
url_errors = (
    y_pred_final != y_test
)

# Predictions from combined model
combined_errors_mask = (
    combined_pred != y_test
)

print("URL-only errors:", url_errors.sum())
print("Combined errors:", combined_errors_mask.sum())

print("\nURL-only correct:", (~url_errors).sum())
print("Combined correct:", (~combined_errors_mask).sum())

URL-only errors: 86
Combined errors: 86

URL-only correct: 12405
Combined correct: 12405


In [126]:
# Cases where URL-only was wrong but combined model became correct
dns_rescued = (
    url_errors &
    ~combined_errors_mask
)

# Cases where URL-only was correct but combined model became wrong
dns_hurt = (
    ~url_errors &
    combined_errors_mask
)

print("Cases DNS/combined model rescued:", dns_rescued.sum())
print("Cases combined model changed correct → wrong:", dns_hurt.sum())

Cases DNS/combined model rescued: 29
Cases combined model changed correct → wrong: 29


In [127]:
!pip install -q dnspython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 6.9 MB/s eta 0:00:00


In [128]:
import dns.resolver
import tldextract

In [129]:
import dns.resolver
import socket

import dns.resolver
import tldextract
from urllib.parse import urlparse


def get_live_dns_features(url):

    # -----------------------------------
    # Extract hostname
    # -----------------------------------
    parsed = urlparse(url)
    hostname = parsed.hostname

    if not hostname:
        return {
            "dns_resolvable": 0,
            "num_ips": 0,
            "ttl": -1,
            "num_name_servers": 0
        }

    # -----------------------------------
    # Extract registered/root domain
    # -----------------------------------
    ext = tldextract.extract(url)

    registered_domain = ext.registered_domain

    if not registered_domain:
        registered_domain = hostname

    resolver = dns.resolver.Resolver()

    dns_resolvable = 0
    num_ips = 0
    ttl = -1
    num_name_servers = 0

    # -----------------------------------
    # A records
    # -----------------------------------
    try:

        answer = resolver.resolve(
            hostname,
            "A"
        )

        ips = set(
            record.address
            for record in answer
        )

        num_ips = len(ips)

        if answer.rrset is not None:
            ttl = answer.rrset.ttl

        if num_ips > 0:
            dns_resolvable = 1

    except Exception:
        pass

    # -----------------------------------
    # NS records
    # Query registered domain, NOT hostname
    # -----------------------------------
    try:

        ns_answer = resolver.resolve(
            registered_domain,
            "NS"
        )

        num_name_servers = len(
            list(ns_answer)
        )

    except Exception:
        pass

    return {
        "dns_resolvable": dns_resolvable,
        "num_ips": num_ips,
        "ttl": ttl,
        "num_name_servers": num_name_servers
    }

In [130]:
test_dns_urls = [
    "https://www.google.com",
    "https://www.github.com",
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.amazon.com",
    "https://www.wikipedia.org",
    "https://www.python.org"
]

for url in test_dns_urls:

    print("\nURL:", url)

    dns_info = get_live_dns_features(url)

    print(dns_info)


URL: https://www.google.com
{'dns_resolvable': 1, 'num_ips': 8, 'ttl': 300, 'num_name_servers': 4}

URL: https://www.github.com
{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 60, 'num_name_servers': 8}

URL: https://www.microsoft.com
{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 20, 'num_name_servers': 4}

URL: https://www.apple.com
{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 20, 'num_name_servers': 4}

URL: https://www.amazon.com


/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 60, 'num_name_servers': 4}

URL: https://www.wikipedia.org
{'dns_resolvable': 1, 'num_ips': 1, 'ttl': 159, 'num_name_servers': 3}

URL: https://www.python.org
{'dns_resolvable': 1, 'num_ips': 4, 'ttl': 37, 'num_name_servers': 4}


In [131]:

live_dns_results = []

for url in unseen_test_urls:

    dns_info = get_live_dns_features(url)

    live_dns_results.append({
        "URL": url,
        **dns_info
    })

live_dns_df = pd.DataFrame(
    live_dns_results
)

display(live_dns_df)

/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


,URL,dns_resolvable,num_ips,ttl,num_name_servers
0,https://www.microsoft.com,1,1,18,4
1,https://www.apple.com,1,1,18,4
2,https://www.amazon.com,1,1,58,4
3,https://www.wikipedia.org,1,1,1,3
4,https://www.python.org,1,4,14,4
5,https://stackoverflow.com,1,1,300,2
6,https://www.linkedin.com,1,2,143,8
7,https://www.netflix.com,1,2,60,4
8,http://secure-account-login-verification-92837...,0,0,-1,0
9,http://paypal-account-security-check-48291.com...,0,0,-1,0


In [132]:
test_subdomain_urls = [
    "https://www.google.com",
    "https://mail.google.com",
    "https://accounts.google.com",
    "https://a.b.example.com",
    "https://www.chevrolet.com.ar"
]

for url in test_subdomain_urls:

    ext = tldextract.extract(url)

    print("\nURL:", url)
    print("Hostname:", urlparse(url).hostname)
    print("Registered domain:", ext.registered_domain)
    print("Subdomain:", ext.subdomain)

    print(
        "DNS:",
        get_live_dns_features(url)
    )


URL: https://www.google.com
Hostname: www.google.com
Registered domain: google.com
Subdomain: www
DNS: {'dns_resolvable': 1, 'num_ips': 8, 'ttl': 296, 'num_name_servers': 4}

URL: https://mail.google.com
Hostname: mail.google.com
Registered domain: google.com
Subdomain: mail
DNS: {'dns_resolvable': 1, 'num_ips': 4, 'ttl': 300, 'num_name_servers': 4}

URL: https://accounts.google.com
Hostname: accounts.google.com
Registered domain: google.com
Subdomain: accounts
DNS: {'dns_resolvable': 1, 'num_ips': 1, 'ttl': 300, 'num_name_servers': 4}

URL: https://a.b.example.com
Hostname: a.b.example.com
Registered domain: example.com
Subdomain: a.b
DNS: {'dns_resolvable': 0, 'num_ips': 0, 'ttl': -1, 'num_name_servers': 2}

URL: https://www.chevrolet.com.ar
Hostname: www.chevrolet.com.ar
Registered domain: chevrolet.com.ar
Subdomain: www
DNS: {'dns_resolvable': 1, 'num_ips': 2, 'ttl': 20, 'num_name_servers': 4}


/tmp/ipykernel_1303/3914519048.py:15: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  print("Registered domain:", ext.registered_domain)
/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


In [133]:
!pip install -q requests

In [134]:
import requests
from datetime import datetime, timezone

In [135]:
import requests

RDAP_BOOTSTRAP_URL = "https://data.iana.org/rdap/dns.json"

rdap_bootstrap = requests.get(
    RDAP_BOOTSTRAP_URL,
    timeout=10
).json()

print("RDAP bootstrap loaded.")

RDAP bootstrap loaded.


In [136]:
def get_rdap_server(domain):

    tld = domain.split(".")[-1].lower()

    services = rdap_bootstrap.get("services", [])

    for service in services:

        tlds = service[0]
        urls = service[1]

        if tld in tlds:
            return urls[0]

    return None

In [137]:
test_domains = [
    "google.com",
    "github.com",
    "python.org",
    "google.co.uk",
    "example.in"
]

for domain in test_domains:

    print(
        domain,
        "→",
        get_rdap_server(domain)
    )

google.com → https://rdap.verisign.com/com/v1/
github.com → https://rdap.verisign.com/com/v1/
python.org → https://rdap.publicinterestregistry.org/rdap/
google.co.uk → https://rdap.nominet.uk/uk/
example.in → https://rdap.nixiregistry.in/rdap/


In [138]:
def get_domain_registration_date(domain):

    rdap_server = get_rdap_server(domain)

    if not rdap_server:
        return None

    url = (
        rdap_server.rstrip("/")
        + "/domain/"
        + domain
    )

    try:

        response = requests.get(
            url,
            timeout=10,
            headers={
                "Accept": "application/rdap+json"
            }
        )

        if response.status_code != 200:
            return None

        data = response.json()

    except Exception:
        return None


    # Look for registration event
    events = data.get("events", [])

    for event in events:

        if event.get("eventAction") == "registration":

            registration_date = event.get(
                "eventDate"
            )

            if registration_date:
                return registration_date

    return None

In [139]:
def get_domain_age(domain):

    registration_date = (
        get_domain_registration_date(domain)
    )

    if not registration_date:
        return -1

    try:

        registration_dt = datetime.fromisoformat(
            registration_date.replace("Z", "+00:00")
        )

        now = datetime.now(timezone.utc)

        age_days = (
            now - registration_dt
        ).days

        return max(age_days, 0)

    except Exception:
        return -1

In [140]:
test_domains = [
    "google.com",
    "github.com",
    "microsoft.com",
    "apple.com",
    "python.org"
]

for domain in test_domains:

    age = get_domain_age(domain)

    print(
        domain,
        "→",
        age,
        "days"
    )

google.com → 10606 days
github.com → 6929 days
microsoft.com → 12934 days
apple.com → 14467 days
python.org → 11509 days


In [141]:
def get_live_domain_intelligence(url):

    parsed = urlparse(url)

    hostname = parsed.hostname

    if not hostname:
        return {
            "domain_age": -1,
            "dns_resolvable": 0,
            "num_ips": 0,
            "ttl": -1,
            "num_name_servers": 0
        }

    ext = tldextract.extract(url)

    registered_domain = (
        ext.top_domain_under_public_suffix
    )

    dns_info = get_live_dns_features(url)

    domain_age = get_domain_age(
        registered_domain
    )

    return {
        "domain_age": domain_age,
        **dns_info
    }

In [142]:
test_urls = [
    "https://www.google.com",
    "https://www.github.com",
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.python.org",
    "https://www.chevrolet.com.ar"
]

for url in test_urls:

    print("\nURL:", url)

    info = get_live_domain_intelligence(url)

    print(info)


URL: https://www.google.com
{'domain_age': 10606, 'dns_resolvable': 1, 'num_ips': 8, 'ttl': 300, 'num_name_servers': 4}

URL: https://www.github.com


/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


{'domain_age': 6929, 'dns_resolvable': 1, 'num_ips': 1, 'ttl': 9, 'num_name_servers': 8}

URL: https://www.microsoft.com
{'domain_age': 12934, 'dns_resolvable': 1, 'num_ips': 1, 'ttl': 20, 'num_name_servers': 4}

URL: https://www.apple.com
{'domain_age': 14467, 'dns_resolvable': 1, 'num_ips': 1, 'ttl': 20, 'num_name_servers': 4}

URL: https://www.python.org
{'domain_age': 11509, 'dns_resolvable': 1, 'num_ips': 4, 'ttl': 29, 'num_name_servers': 4}

URL: https://www.chevrolet.com.ar
{'domain_age': -1, 'dns_resolvable': 1, 'num_ips': 2, 'ttl': 20, 'num_name_servers': 4}


In [143]:
# ============================================
# LIVE DOMAIN INTELLIGENCE - VALIDATION TEST
# ============================================

test_urls = [
    # 1. Normal well-known domains
    "https://www.google.com",
    "https://www.github.com",
    "https://www.microsoft.com",
    "https://www.apple.com",
    "https://www.python.org",
    "https://www.wikipedia.org",

    # 2. Subdomains
    "https://mail.google.com",
    "https://accounts.google.com",
    "https://docs.google.com",

    # 3. URL with path
    "https://www.github.com/login",

    # 4. URL with query parameters
    "https://www.google.com/search?q=python",

    # 5. Domain without www
    "https://github.com",

    # 6. Safe synthetic non-resolving domain
    "https://this-domain-should-not-exist-928374928374.com"
]


for url in test_urls:

    print("=" * 80)
    print("URL:", url)

    try:
        info = get_live_domain_intelligence(url)

        print("Domain Age       :", info["domain_age"], "days")
        print("DNS Resolvable   :", info["dns_resolvable"])
        print("Number of IPs    :", info["num_ips"])
        print("TTL              :", info["ttl"])
        print("Name Servers     :", info["num_name_servers"])

    except Exception as e:
        print("ERROR:", e)

URL: https://www.google.com
Domain Age       : 10606 days
DNS Resolvable   : 1
Number of IPs    : 8
TTL              : 275
Name Servers     : 4
URL: https://www.github.com


/tmp/ipykernel_1303/4118925904.py:30: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  registered_domain = ext.registered_domain


Domain Age       : 6929 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 60
Name Servers     : 8
URL: https://www.microsoft.com
Domain Age       : 12934 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 20
Name Servers     : 4
URL: https://www.apple.com
Domain Age       : 14467 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 12
Name Servers     : 4
URL: https://www.python.org
Domain Age       : 11509 days
DNS Resolvable   : 1
Number of IPs    : 4
TTL              : 4
Name Servers     : 4
URL: https://www.wikipedia.org
Domain Age       : 9390 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 91
Name Servers     : 3
URL: https://mail.google.com
Domain Age       : 10606 days
DNS Resolvable   : 1
Number of IPs    : 4
TTL              : 300
Name Servers     : 4
URL: https://accounts.google.com
Domain Age       : 10606 days
DNS Resolvable   : 1
Number of IPs    : 1
TTL              : 300
Name Servers     : 4
URL: https://docs

In [144]:
# ============================================
# LIVE DOMAIN INTELLIGENCE
# DNS + RDAP DOMAIN AGE
# ============================================

import requests
import dns.resolver
import tldextract

from datetime import datetime, timezone
from urllib.parse import urlparse


# --------------------------------------------
# 1. Load IANA RDAP bootstrap
# --------------------------------------------

RDAP_BOOTSTRAP_URL = "https://data.iana.org/rdap/dns.json"

rdap_bootstrap = requests.get(
    RDAP_BOOTSTRAP_URL,
    timeout=10
).json()


# --------------------------------------------
# 2. Find RDAP server for a TLD
# --------------------------------------------

def get_rdap_server(domain):

    tld = domain.split(".")[-1].lower()

    services = rdap_bootstrap.get("services", [])

    for service in services:

        tlds = service[0]
        urls = service[1]

        if tld in tlds:
            return urls[0]

    return None


# --------------------------------------------
# 3. Get domain registration date
# --------------------------------------------

def get_domain_registration_date(domain):

    rdap_server = get_rdap_server(domain)

    if not rdap_server:
        return None

    url = (
        rdap_server.rstrip("/")
        + "/domain/"
        + domain
    )

    try:

        response = requests.get(
            url,
            timeout=10,
            headers={
                "Accept": "application/rdap+json"
            }
        )

        if response.status_code != 200:
            return None

        data = response.json()

    except Exception:
        return None


    for event in data.get("events", []):

        if event.get("eventAction") == "registration":

            registration_date = event.get("eventDate")

            if registration_date:
                return registration_date

    return None


# --------------------------------------------
# 4. Calculate domain age
# --------------------------------------------

def get_domain_age(domain):

    registration_date = get_domain_registration_date(domain)

    if not registration_date:
        return -1

    try:

        registration_dt = datetime.fromisoformat(
            registration_date.replace(
                "Z",
                "+00:00"
            )
        )

        now = datetime.now(timezone.utc)

        age_days = (
            now - registration_dt
        ).days

        return max(age_days, 0)

    except Exception:
        return -1


# --------------------------------------------
# 5. Get live DNS information
# --------------------------------------------

def get_live_dns_features(url):

    parsed = urlparse(url)

    hostname = parsed.hostname

    # Invalid URL
    if not hostname:

        return {
            "dns_resolvable": 0,
            "num_ips": 0,
            "ttl": -1,
            "num_name_servers": 0
        }


    # Extract registered/apex domain
    ext = tldextract.extract(url)

    # IMPORTANT:
    # Use the new property instead of
    # deprecated registered_domain
    registered_domain = (
        ext.top_domain_under_public_suffix
    )

    if not registered_domain:
        registered_domain = hostname


    resolver = dns.resolver.Resolver()

    dns_resolvable = 0
    num_ips = 0
    ttl = -1
    num_name_servers = 0


    # ----------------------------------------
    # A record
    # ----------------------------------------

    try:

        answer = resolver.resolve(
            hostname,
            "A"
        )

        ips = set(
            record.address
            for record in answer
        )

        num_ips = len(ips)

        if answer.rrset is not None:
            ttl = answer.rrset.ttl

        if num_ips > 0:
            dns_resolvable = 1

    except Exception:
        pass


    # ----------------------------------------
    # NS records
    # Query registered domain, NOT hostname
    # ----------------------------------------

    try:

        ns_answer = resolver.resolve(
            registered_domain,
            "NS"
        )

        num_name_servers = len(
            list(ns_answer)
        )

    except Exception:
        pass


    return {
        "dns_resolvable": dns_resolvable,
        "num_ips": num_ips,
        "ttl": ttl,
        "num_name_servers": num_name_servers
    }


# --------------------------------------------
# 6. Complete live domain intelligence
# --------------------------------------------

def get_live_domain_intelligence(url):

    parsed = urlparse(url)

    hostname = parsed.hostname

    # Invalid URL
    if not hostname:

        return {
            "domain_age": -1,
            "dns_resolvable": 0,
            "num_ips": 0,
            "ttl": -1,
            "num_name_servers": 0
        }


    # Extract registered domain
    ext = tldextract.extract(url)

    registered_domain = (
        ext.top_domain_under_public_suffix
    )

    if not registered_domain:
        registered_domain = hostname


    # Get live DNS
    dns_info = get_live_dns_features(url)


    # Get live domain age
    domain_age = get_domain_age(
        registered_domain
    )


    return {
        "domain_age": domain_age,
        **dns_info
    }


print("Live DNS + RDAP intelligence functions loaded successfully.")

Live DNS + RDAP intelligence functions loaded successfully.


In [145]:
# Quick verification

test_url = "https://www.github.com"

print("URL:", test_url)

info = get_live_domain_intelligence(test_url)

print(info)

URL: https://www.github.com
{'domain_age': 6929, 'dns_resolvable': 1, 'num_ips': 1, 'ttl': 5, 'num_name_servers': 8}


In [146]:
# ============================================
# COMPLETE URL ANALYSIS
# ML + LIVE DNS/RDAP INTELLIGENCE
# ============================================

def analyze_url(url):

    # ----------------------------------------
    # 1. ML prediction
    # ----------------------------------------

    ml_result = predict_url(url)


    # ----------------------------------------
    # 2. Live DNS + RDAP intelligence
    # ----------------------------------------

    live_result = get_live_domain_intelligence(url)


    # ----------------------------------------
    # 3. Combine everything
    # ----------------------------------------

    result = {
        "url": url,

        # ML results
        "prediction": ml_result["prediction"],
        "phishing_probability": ml_result["phishing_probability"],
        "legitimate_probability": ml_result["legitimate_probability"],

        # Live intelligence
        "domain_age": live_result["domain_age"],
        "dns_resolvable": live_result["dns_resolvable"],
        "num_ips": live_result["num_ips"],
        "ttl": live_result["ttl"],
        "num_name_servers": live_result["num_name_servers"]
    }

    return result

In [147]:
# ============================================
# TEST COMPLETE ANALYSIS
# ============================================

test_url = "https://www.google.com"

result = analyze_url(test_url)

result

{'url': 'https://www.google.com',
 'prediction': 'Legitimate',
 'phishing_probability': np.float32(0.005525957),
 'legitimate_probability': np.float32(0.99447405),
 'domain_age': 10606,
 'dns_resolvable': 1,
 'num_ips': 8,
 'ttl': 300,
 'num_name_servers': 4}

In [148]:
test_url = "http://secure-login-account-verification-12345.com/login"

result = analyze_url(test_url)

result

{'url': 'http://secure-login-account-verification-12345.com/login',
 'prediction': 'Phishing',
 'phishing_probability': np.float32(0.93720645),
 'legitimate_probability': np.float32(0.06279355),
 'domain_age': -1,
 'dns_resolvable': 0,
 'num_ips': 0,
 'ttl': -1,
 'num_name_servers': 0}

In [150]:
# ============================================
# RISK & EXPLANATION LAYER
# ============================================

def generate_explanation(result):

    explanation = {}

    # ----------------------------------------
    # 1. ML prediction
    # ----------------------------------------

    prediction = result["prediction"]

    phishing_probability = float(
        result["phishing_probability"]
    )

    legitimate_probability = float(
        result["legitimate_probability"]
    )

    explanation["prediction"] = prediction

    explanation["phishing_probability"] = round(
        phishing_probability * 100,
        2
    )

    explanation["legitimate_probability"] = round(
        legitimate_probability * 100,
        2
    )


    # ----------------------------------------
    # 2. Risk level
    # ----------------------------------------

    if phishing_probability >= 0.80:

        risk_level = "High"

    elif phishing_probability >= 0.50:

        risk_level = "Medium"

    else:

        risk_level = "Low"


    explanation["risk_level"] = risk_level


    # ----------------------------------------
    # 3. Live DNS/RDAP information
    # ----------------------------------------

    explanation["domain_age_days"] = result[
        "domain_age"
    ]

    explanation["dns_resolvable"] = (
        "Yes"
        if result["dns_resolvable"] == 1
        else "No"
    )

    explanation["num_ips"] = result[
        "num_ips"
    ]

    explanation["ttl"] = result[
        "ttl"
    ]

    explanation["num_name_servers"] = result[
        "num_name_servers"
    ]


    # ----------------------------------------
    # 4. Human-readable evidence
    # ----------------------------------------

    evidence = []


    # DNS evidence
    if result["dns_resolvable"] == 0:

        evidence.append(
            "The hostname could not be resolved through DNS."
        )

    else:

        evidence.append(
            "The hostname successfully resolves through DNS."
        )


    # Domain age evidence
    if result["domain_age"] == -1:

        evidence.append(
            "Domain registration age was unavailable."
        )

    else:

        evidence.append(
            f"The domain has existed for approximately "
            f"{result['domain_age']} days."
        )


    # IP evidence
    if result["num_ips"] == 0:

        evidence.append(
            "No IP addresses were returned."
        )

    else:

        evidence.append(
            f"{result['num_ips']} IP address(es) were returned."
        )


    explanation["evidence"] = evidence


    # ----------------------------------------
    # 5. Final interpretation
    # ----------------------------------------

    if prediction == "Phishing":

        explanation["summary"] = (
            "The machine-learning model classified this "
            "URL as phishing based on the URL patterns "
            "learned from the training data."
        )

    else:

        explanation["summary"] = (
            "The machine-learning model classified this "
            "URL as legitimate based on the URL patterns "
            "learned from the training data."
        )


    # ----------------------------------------
    # 6. Important limitation
    # ----------------------------------------

    explanation["note"] = (
        "Live DNS and domain information is supporting "
        "evidence. It does not by itself prove that a "
        "URL is malicious or legitimate."
    )


    return explanation

In [151]:
explanation = generate_explanation(result)

explanation

{'prediction': 'Phishing',
 'phishing_probability': 93.72,
 'legitimate_probability': 6.28,
 'risk_level': 'High',
 'domain_age_days': -1,
 'dns_resolvable': 'No',
 'num_ips': 0,
 'ttl': -1,
 'num_name_servers': 0,
 'evidence': ['The hostname could not be resolved through DNS.',
  'Domain registration age was unavailable.',
  'No IP addresses were returned.'],
 'summary': 'The machine-learning model classified this URL as phishing based on the URL patterns learned from the training data.',
 'note': 'Live DNS and domain information is supporting evidence. It does not by itself prove that a URL is malicious or legitimate.'}

In [152]:
# ============================================
# SHAP SETUP
# ============================================

!pip install -q shap

import shap

print("SHAP version:", shap.__version__)
print("SHAP loaded successfully.")

SHAP version: 0.52.0
SHAP loaded successfully.


In [153]:
# ============================================
# SHAP EXPLANATION FOR ONE URL
# ============================================

# URL we want to explain
test_url = "http://secure-login-account-verification-12345.com/login"


# --------------------------------------------
# 1. Extract URL features
# --------------------------------------------

features = extract_url_features(test_url)

X_explain = pd.DataFrame(
    [features],
    columns=url_features_final
)

print("Extracted Features:")
display(X_explain)


# --------------------------------------------
# 2. Create SHAP TreeExplainer
# --------------------------------------------

explainer = shap.TreeExplainer(final_xgb_model)

shap_values = explainer.shap_values(X_explain)


# --------------------------------------------
# 3. Display SHAP values
# --------------------------------------------

shap_df = pd.DataFrame({
    "feature": url_features_final,
    "value": X_explain.iloc[0].values,
    "shap_value": shap_values[0]
})

# Sort by absolute contribution
shap_df["absolute_shap"] = shap_df["shap_value"].abs()

shap_df = shap_df.sort_values(
    "absolute_shap",
    ascending=False
)

print("\nFeature Contributions:")
display(shap_df)

Extracted Features:


,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,num_dots,num_hyphens,ratio_digits_host,num_question_mark,num_eq
0,4.40845,56,0,0.089286,0,43,1,4,0.116279,0,0



Feature Contributions:


,feature,value,shap_value,absolute_shap
1,url_length,56.000000,5.488614,5.488614
5,length_hostname,43.000000,-2.531798,2.531798
0,entropy,4.408450,1.003412,1.003412
6,num_dots,1.000000,-0.931066,0.931066
4,num_subdomains,0.000000,-0.787012,0.787012
3,ratio_digits_url,0.089286,0.622482,0.622482
7,num_hyphens,4.000000,-0.303168,0.303168
2,num_www,0.000000,0.272731,0.272731
8,ratio_digits_host,0.116279,-0.250589,0.250589
10,num_eq,0.000000,-0.002824,0.002824


In [154]:
# ============================================
# AUTOMATIC SHAP FEATURE EXPLANATION
# ============================================

# Create the SHAP explainer once
shap_explainer = shap.TreeExplainer(final_xgb_model)


def generate_feature_explanation(url):

    # ----------------------------------------
    # 1. Extract URL features
    # ----------------------------------------

    features = extract_url_features(url)

    X_input = pd.DataFrame(
        [features],
        columns=url_features_final
    )


    # ----------------------------------------
    # 2. Get SHAP values
    # ----------------------------------------

    shap_values = shap_explainer.shap_values(X_input)

    shap_values = shap_values[0]


    # ----------------------------------------
    # 3. Create explanation table
    # ----------------------------------------

    explanation_df = pd.DataFrame({
        "feature": url_features_final,
        "value": X_input.iloc[0].values,
        "shap_value": shap_values
    })


    # ----------------------------------------
    # 4. Separate positive and negative
    # ----------------------------------------

    positive_features = explanation_df[
        explanation_df["shap_value"] > 0
    ].copy()

    negative_features = explanation_df[
        explanation_df["shap_value"] < 0
    ].copy()


    # Sort strongest first
    positive_features = positive_features.sort_values(
        "shap_value",
        ascending=False
    )

    negative_features = negative_features.sort_values(
        "shap_value",
        ascending=True
    )


    # ----------------------------------------
    # 5. Take strongest contributors
    # ----------------------------------------

    top_positive = positive_features.head(3)

    top_negative = negative_features.head(3)


    # ----------------------------------------
    # 6. Create readable explanations
    # ----------------------------------------

    positive_explanations = []

    for _, row in top_positive.iterrows():

        feature = row["feature"]
        value = row["value"]

        sentence = (
            f"{feature} = {value:.4f} "
            f"pushed the prediction toward phishing."
        )

        positive_explanations.append(sentence)


    negative_explanations = []

    for _, row in top_negative.iterrows():

        feature = row["feature"]
        value = row["value"]

        sentence = (
            f"{feature} = {value:.4f} "
            f"pushed the prediction toward legitimate."
        )

        negative_explanations.append(sentence)


    # ----------------------------------------
    # 7. Return everything
    # ----------------------------------------

    return {
        "positive_contributors": positive_explanations,
        "negative_contributors": negative_explanations,
        "shap_table": explanation_df.sort_values(
            "shap_value",
            key=lambda x: x.abs(),
            ascending=False
        )
    }

In [155]:
test_url = "http://secure-login-account-verification-12345.com/login"

feature_explanation = generate_feature_explanation(test_url)

print("========================================")
print("FEATURES PUSHING TOWARD PHISHING")
print("========================================")

for item in feature_explanation["positive_contributors"]:
    print("•", item)


print("\n========================================")
print("FEATURES PUSHING TOWARD LEGITIMATE")
print("========================================")

for item in feature_explanation["negative_contributors"]:
    print("•", item)


print("\n========================================")
print("FULL SHAP TABLE")
print("========================================")

display(feature_explanation["shap_table"])

FEATURES PUSHING TOWARD PHISHING
• url_length = 56.0000 pushed the prediction toward phishing.
• entropy = 4.4084 pushed the prediction toward phishing.
• ratio_digits_url = 0.0893 pushed the prediction toward phishing.

FEATURES PUSHING TOWARD LEGITIMATE
• length_hostname = 43.0000 pushed the prediction toward legitimate.
• num_dots = 1.0000 pushed the prediction toward legitimate.
• num_subdomains = 0.0000 pushed the prediction toward legitimate.

FULL SHAP TABLE


,feature,value,shap_value
1,url_length,56.000000,5.488614
5,length_hostname,43.000000,-2.531798
0,entropy,4.408450,1.003412
6,num_dots,1.000000,-0.931066
4,num_subdomains,0.000000,-0.787012
3,ratio_digits_url,0.089286,0.622482
7,num_hyphens,4.000000,-0.303168
2,num_www,0.000000,0.272731
8,ratio_digits_host,0.116279,-0.250589
10,num_eq,0.000000,-0.002824


In [156]:
# ============================================
# HUMAN-READABLE FEATURE DESCRIPTIONS
# ============================================

feature_descriptions = {

    "entropy":
        "URL character entropy",

    "url_length":
        "URL length",

    "num_www":
        "Number of 'www' occurrences",

    "ratio_digits_url":
        "Digit ratio in the URL",

    "num_subdomains":
        "Number of subdomains",

    "length_hostname":
        "Hostname length",

    "num_dots":
        "Number of dots in the URL",

    "num_hyphens":
        "Number of hyphens in the URL",

    "ratio_digits_host":
        "Digit ratio in the hostname",

    "num_question_mark":
        "Number of question marks",

    "num_eq":
        "Number of '=' characters"
}

In [157]:
# ============================================
# HUMAN-READABLE SHAP EXPLANATION
# ============================================

def format_feature_value(feature, value):

    if feature == "url_length":
        return f"{int(value)} characters"

    elif feature == "entropy":
        return f"{value:.2f}"

    elif feature == "ratio_digits_url":
        return f"{value * 100:.2f}%"

    elif feature == "ratio_digits_host":
        return f"{value * 100:.2f}%"

    elif feature in [
        "num_www",
        "num_subdomains",
        "length_hostname",
        "num_dots",
        "num_hyphens",
        "num_question_mark",
        "num_eq"
    ]:
        return f"{int(value)}"

    else:
        return f"{value:.2f}"


def generate_readable_feature_explanation(url):

    # Get SHAP explanation
    result = generate_feature_explanation(url)

    shap_table = result["shap_table"]


    # ----------------------------------------
    # Strongest phishing contributors
    # ----------------------------------------

    positive = shap_table[
        shap_table["shap_value"] > 0
    ].sort_values(
        "shap_value",
        ascending=False
    ).head(3)


    # ----------------------------------------
    # Strongest legitimate contributors
    # ----------------------------------------

    negative = shap_table[
        shap_table["shap_value"] < 0
    ].sort_values(
        "shap_value",
        ascending=True
    ).head(3)


    # ----------------------------------------
    # Convert to readable sentences
    # ----------------------------------------

    phishing_reasons = []

    for _, row in positive.iterrows():

        feature = row["feature"]
        value = row["value"]

        readable_name = feature_descriptions[feature]

        readable_value = format_feature_value(
            feature,
            value
        )

        phishing_reasons.append(
            f"{readable_name}: {readable_value} "
            f"(contributed toward the phishing prediction)"
        )


    legitimate_reasons = []

    for _, row in negative.iterrows():

        feature = row["feature"]
        value = row["value"]

        readable_name = feature_descriptions[feature]

        readable_value = format_feature_value(
            feature,
            value
        )

        legitimate_reasons.append(
            f"{readable_name}: {readable_value} "
            f"(contributed toward the legitimate prediction)"
        )


    return {
        "phishing_reasons": phishing_reasons,
        "legitimate_reasons": legitimate_reasons,
        "shap_table": shap_table
    }

In [158]:
test_url = "http://secure-login-account-verification-12345.com/login"

readable_explanation = generate_readable_feature_explanation(
    test_url
)

print("========================================")
print("WHY THE MODEL LEANED TOWARD PHISHING")
print("========================================")

for reason in readable_explanation["phishing_reasons"]:
    print("•", reason)


print("\n========================================")
print("FEATURES PUSHING THE OTHER WAY")
print("========================================")

for reason in readable_explanation["legitimate_reasons"]:
    print("•", reason)

WHY THE MODEL LEANED TOWARD PHISHING
• URL length: 56 characters (contributed toward the phishing prediction)
• URL character entropy: 4.41 (contributed toward the phishing prediction)
• Digit ratio in the URL: 8.93% (contributed toward the phishing prediction)

FEATURES PUSHING THE OTHER WAY
• Hostname length: 43 (contributed toward the legitimate prediction)
• Number of dots in the URL: 1 (contributed toward the legitimate prediction)
• Number of subdomains: 0 (contributed toward the legitimate prediction)


In [159]:
# ============================================
# FINAL URL ANALYSIS PIPELINE
# ML + SHAP + DNS + RDAP
# ============================================

def analyze_url_complete(url):

    # ----------------------------------------
    # 1. ML prediction + live intelligence
    # ----------------------------------------

    result = analyze_url(url)


    # ----------------------------------------
    # 2. Risk level
    # ----------------------------------------

    phishing_probability = float(
        result["phishing_probability"]
    )

    if phishing_probability >= 0.80:
        risk_level = "High"

    elif phishing_probability >= 0.50:
        risk_level = "Medium"

    else:
        risk_level = "Low"


    # ----------------------------------------
    # 3. SHAP explanation
    # ----------------------------------------

    feature_explanation = (
        generate_readable_feature_explanation(url)
    )


    # ----------------------------------------
    # 4. Combine everything
    # ----------------------------------------

    complete_result = {

        # Basic information
        "url": url,

        # ML
        "prediction": result["prediction"],

        "phishing_probability": round(
            float(result["phishing_probability"]) * 100,
            2
        ),

        "legitimate_probability": round(
            float(result["legitimate_probability"]) * 100,
            2
        ),

        "risk_level": risk_level,


        # SHAP explanations
        "phishing_reasons": (
            feature_explanation[
                "phishing_reasons"
            ]
        ),

        "legitimate_reasons": (
            feature_explanation[
                "legitimate_reasons"
            ]
        ),


        # Live DNS/RDAP
        "domain_age_days": result[
            "domain_age"
        ],

        "dns_resolvable": (
            "Yes"
            if result["dns_resolvable"] == 1
            else "No"
        ),

        "num_ips": result[
            "num_ips"
        ],

        "ttl": result[
            "ttl"
        ],

        "num_name_servers": result[
            "num_name_servers"
        ]
    }


    return complete_result

In [161]:
test_url = "http://secure-login-account-verification-12345.com/login"

final_result = analyze_url_complete(test_url)

final_result

{'url': 'http://secure-login-account-verification-12345.com/login',
 'prediction': 'Phishing',
 'phishing_probability': 93.72,
 'legitimate_probability': 6.28,
 'risk_level': 'High',
 'phishing_reasons': ['URL length: 56 characters (contributed toward the phishing prediction)',
  'URL character entropy: 4.41 (contributed toward the phishing prediction)',
  'Digit ratio in the URL: 8.93% (contributed toward the phishing prediction)'],
 'legitimate_reasons': ['Hostname length: 43 (contributed toward the legitimate prediction)',
  'Number of dots in the URL: 1 (contributed toward the legitimate prediction)',
  'Number of subdomains: 0 (contributed toward the legitimate prediction)'],
 'domain_age_days': -1,
 'dns_resolvable': 'No',
 'num_ips': 0,
 'ttl': -1,
 'num_name_servers': 0}

In [162]:
# ============================================
# COMPLETE PIPELINE - MULTIPLE URL TEST
# ============================================

test_urls = [
    "https://www.google.com",
    "https://github.com",
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "https://www.google.com/search?q=python"
]

for url in test_urls:

    print("\n" + "=" * 80)
    print("URL:", url)
    print("=" * 80)

    result = analyze_url_complete(url)

    print("Prediction          :", result["prediction"])
    print("Phishing Probability:", result["phishing_probability"], "%")
    print("Risk Level          :", result["risk_level"])

    print("\nTop phishing contributors:")
    for reason in result["phishing_reasons"]:
        print("  •", reason)

    print("\nTop opposing contributors:")
    for reason in result["legitimate_reasons"]:
        print("  •", reason)

    print("\nLive Intelligence:")
    print("  Domain age        :", result["domain_age_days"], "days")
    print("  DNS resolvable    :", result["dns_resolvable"])
    print("  IPs               :", result["num_ips"])
    print("  TTL               :", result["ttl"])
    print("  Name servers      :", result["num_name_servers"])


URL: https://www.google.com
Prediction          : Legitimate
Phishing Probability: 0.55 %
Risk Level          : Low

Top phishing contributors:
  • Number of subdomains: 1 (contributed toward the phishing prediction)
  • Hostname length: 14 (contributed toward the phishing prediction)
  • Digit ratio in the hostname: 0.00% (contributed toward the phishing prediction)

Top opposing contributors:
  • URL length: 22 characters (contributed toward the legitimate prediction)
  • Number of 'www' occurrences: 1 (contributed toward the legitimate prediction)
  • URL character entropy: 3.66 (contributed toward the legitimate prediction)

Live Intelligence:
  Domain age        : 10606 days
  DNS resolvable    : Yes
  IPs               : 8
  TTL               : 300
  Name servers      : 4

URL: https://github.com
Prediction          : Legitimate
Phishing Probability: 0.48 %
Risk Level          : Low

Top phishing contributors:
  • Hostname length: 10 (contributed toward the phishing prediction)


In [163]:
# ============================================
# COMPARE GOOGLE URL FEATURES
# ============================================

urls_to_compare = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://www.google.com/?q=python",
    "https://www.google.com/search?q=test"
]

for url in urls_to_compare:

    features = extract_url_features(url)

    result = predict_url(url)

    print("\n" + "=" * 80)
    print("URL:", url)
    print("Prediction:", result["prediction"])
    print(
        "Phishing probability:",
        float(result["phishing_probability"])
    )

    print("\nFeatures:")

    for feature in url_features_final:
        print(
            f"{feature:20s}: {features[feature]}"
        )


URL: https://www.google.com
Prediction: Legitimate
Phishing probability: 0.005525956861674786

Features:
entropy             : 3.663532754804254
url_length          : 22
num_www             : 1
ratio_digits_url    : 0.0
num_subdomains      : 1
length_hostname     : 14
num_dots            : 2
num_hyphens         : 0
ratio_digits_host   : 0.0
num_question_mark   : 0
num_eq              : 0

URL: https://www.google.com/search
Prediction: Phishing
Phishing probability: 0.9923157691955566

Features:
entropy             : 3.8833374604210062
url_length          : 29
num_www             : 1
ratio_digits_url    : 0.0
num_subdomains      : 1
length_hostname     : 14
num_dots            : 2
num_hyphens         : 0
ratio_digits_host   : 0.0
num_question_mark   : 0
num_eq              : 0

URL: https://www.google.com/search?q=python
Prediction: Phishing
Phishing probability: 0.9997850060462952

Features:
entropy             : 4.221097250057957
url_length          : 38
num_www             : 1
ratio

In [164]:
# ============================================
# CHECK URL PATH / QUERY DISTRIBUTION
# ============================================

import pandas as pd
from urllib.parse import urlparse


def get_path_length(url):
    try:
        return len(urlparse(url).path)
    except:
        return 0


def get_query_length(url):
    try:
        return len(urlparse(url).query)
    except:
        return 0


def has_path(url):
    try:
        return int(urlparse(url).path not in ["", "/"])
    except:
        return 0


def has_query(url):
    try:
        return int(bool(urlparse(url).query))
    except:
        return 0


analysis_df = df_clean.copy()

analysis_df["path_length"] = analysis_df["url"].apply(
    get_path_length
)

analysis_df["query_length"] = analysis_df["url"].apply(
    get_query_length
)

analysis_df["has_path"] = analysis_df["url"].apply(
    has_path
)

analysis_df["has_query"] = analysis_df["url"].apply(
    has_query
)


print("========================================")
print("PATH / QUERY DISTRIBUTION")
print("========================================")

print("\nPath presence:")
print(
    pd.crosstab(
        analysis_df["label"],
        analysis_df["has_path"],
        normalize="index"
    ) * 100
)

print("\nQuery presence:")
print(
    pd.crosstab(
        analysis_df["label"],
        analysis_df["has_query"],
        normalize="index"
    ) * 100
)


print("\nAverage path length:")
print(
    analysis_df.groupby("label")["path_length"].mean()
)


print("\nAverage query length:")
print(
    analysis_df.groupby("label")["query_length"].mean()
)

PATH / QUERY DISTRIBUTION

Path presence:
has_path             0          1
label                            
legitimate  100.000000   0.000000
phishing     33.560098  66.439902

Query presence:
has_query            0          1
label                            
legitimate  100.000000   0.000000
phishing     79.804177  20.195823

Average path length:
label
legitimate     0.000000
phishing      22.365547
Name: path_length, dtype: float64

Average query length:
label
legitimate     0.000000
phishing      13.368185
Name: query_length, dtype: float64


In [165]:
# ============================================
# CHECK FEATURE IMPORTANCE FOR PATH-RELATED
# LEGITIMATE URL PROBLEM
# ============================================

print("Current URL model feature importance:")
print()

importance_df = pd.DataFrame({
    "feature": url_features_final,
    "importance": final_xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

display(importance_df)

Current URL model feature importance:



,feature,importance
1,url_length,0.186196
0,entropy,0.175654
2,num_www,0.152400
4,num_subdomains,0.119720
5,length_hostname,0.093645
3,ratio_digits_url,0.093307
10,num_eq,0.074939
6,num_dots,0.034461
8,ratio_digits_host,0.029617
9,num_question_mark,0.021410


In [169]:
import random
from urllib.parse import urlparse

random.seed(42)

# --------------------------------------------------
# 1. Get legitimate URLs from TRAINING data only
# --------------------------------------------------

train_legitimate = train_df[train_df["label"] == 0].copy()

print("Original legitimate training URLs:", len(train_legitimate))


# --------------------------------------------------
# 2. Extract domains from training legitimate URLs
# --------------------------------------------------

train_legitimate["domain"] = train_legitimate["url"].apply(
    lambda x: urlparse(x).netloc
)

legitimate_domains = (
    train_legitimate["domain"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Unique legitimate training domains:", len(legitimate_domains))


# --------------------------------------------------
# 3. Templates for realistic legitimate URLs
# --------------------------------------------------

path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example",
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2",
]


# --------------------------------------------------
# 4. Generate augmented legitimate URLs
# --------------------------------------------------

augmented_urls = []

# Generate approximately 10,000 additional examples
target_count = 10000

while len(augmented_urls) < target_count:

    domain = random.choice(legitimate_domains)

    # 70% path URLs
    # 30% query URLs
    if random.random() < 0.70:
        path = random.choice(path_templates)
    else:
        path = random.choice(query_templates)

    new_url = "https://" + domain + path

    augmented_urls.append(new_url)


# Remove duplicates
augmented_urls = list(set(augmented_urls))

print("Augmented legitimate URLs:", len(augmented_urls))

# Show examples
augmented_urls[:20]



NameError: name 'train_df' is not defined

In [168]:
[name for name in globals().keys()
 if any(word in name.lower() for word in ["train", "test", "split", "domain"])]

['extract_domain',
 'domain_counts',
 'domain_label_counts',
 'conflicting_domains',
 'domain_label',
 'GroupShuffleSplit',
 'train_idx',
 'test_idx',
 'X_train',
 'X_test',
 'y_train',
 'y_test',
 'groups_train',
 'groups_test',
 'train_domains',
 'test_domains',
 'X_train_no_length',
 'X_test_no_length',
 'X_train_log',
 'X_test_log',
 'test_results',
 'test_url',
 'test_urls',
 'get_num_subdomains',
 'X_train_clean',
 'X_test_clean',
 'unseen_test_urls',
 'domain',
 'domain_exists',
 'domain_age_unknown',
 'X_train_dns_raw',
 'X_test_dns_raw',
 'X_train_dns',
 'X_test_dns',
 'X_train_combined_url',
 'X_test_combined_url',
 'X_train_combined_dns',
 'X_test_combined_dns',
 'X_train_dns_imputed',
 'X_test_dns_imputed',
 'X_train_combined',
 'X_test_combined',
 'test_dns_urls',
 'test_subdomain_urls',
 'get_domain_registration_date',
 'get_domain_age',
 'get_live_domain_intelligence']

In [170]:
import random
from urllib.parse import urlparse

random.seed(42)

# Use the EXISTING domain-based training split
train_data = df_clean.iloc[train_idx].copy()

# Keep only legitimate URLs
train_legitimate = train_data[train_data["label"] == 0].copy()

print("Original legitimate training URLs:", len(train_legitimate))


# Extract domains
train_legitimate["domain"] = train_legitimate["url"].apply(
    lambda x: urlparse(x).netloc
)

legitimate_domains = (
    train_legitimate["domain"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Unique legitimate training domains:", len(legitimate_domains))


# URL structures that legitimate websites commonly use
path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example",
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2",
]


# Generate additional legitimate examples
augmented_urls = []

target_count = 10000

while len(augmented_urls) < target_count:

    domain = random.choice(legitimate_domains)

    if random.random() < 0.70:
        path = random.choice(path_templates)
    else:
        path = random.choice(query_templates)

    new_url = "https://" + domain + path

    augmented_urls.append(new_url)


# Remove duplicates
augmented_urls = list(set(augmented_urls))

print("Augmented legitimate URLs:", len(augmented_urls))

print("\nSample augmented URLs:")
for url in augmented_urls[:20]:
    print(url)

Original legitimate training URLs: 0
Unique legitimate training domains: 0


IndexError: Cannot choose from an empty sequence

In [172]:
import numpy as np

print("df_clean shape:", df_clean.shape)

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\ny_train values:")
print(np.unique(y_train, return_counts=True))

print("\ny_test values:")
print(np.unique(y_test, return_counts=True))

print("\ntrain_idx length:", len(train_idx))
print("test_idx length:", len(test_idx))

print("\nFirst 10 train_idx:")
print(train_idx[:10])

print("\nFirst 10 df_clean indices:")
print(df_clean.index[:10].tolist())

print("\nFirst 10 X_train indices:")
print(X_train.index[:10].tolist())

df_clean shape: (59814, 23)

X_train shape: (47323, 13)
X_test shape: (12491, 13)

y_train shape: (47323,)
y_test shape: (12491,)

y_train values:
(array([0, 1]), array([22673, 24650]))

y_test values:
(array([0, 1]), array([5684, 6807]))

train_idx length: 47323
test_idx length: 12491

First 10 train_idx:
[ 0  2  3  4  5  6  7  8 10 11]

First 10 df_clean indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

First 10 X_train indices:
[0, 2, 3, 4, 5, 6, 7, 8, 10, 11]


In [173]:
print(df_clean["label"].value_counts())
print("\nLabel data types:")
print(df_clean["label"].map(type).value_counts())

print("\nSample labels:")
print(df_clean["label"].head(10).tolist())

label
phishing      31457
legitimate    28357
Name: count, dtype: int64

Label data types:
label
<class 'str'>    59814
Name: count, dtype: int64

Sample labels:
['legitimate', 'phishing', 'phishing', 'phishing', 'legitimate', 'legitimate', 'legitimate', 'legitimate', 'phishing', 'phishing']


In [174]:
# Get the exact training rows using the existing train_idx
train_data = df_clean.iloc[train_idx].copy()

# Select legitimate rows using the actual string label
train_legitimate = train_data[
    train_data["label"] == "legitimate"
].copy()

print("Original legitimate training URLs:", len(train_legitimate))

# Extract domains
train_legitimate["domain"] = train_legitimate["url"].apply(
    lambda x: urlparse(x).netloc
)

legitimate_domains = (
    train_legitimate["domain"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Unique legitimate training domains:", len(legitimate_domains))

print("\nSample legitimate URLs:")
print(train_legitimate["url"].head(10).tolist())

Original legitimate training URLs: 22673
Unique legitimate training domains: 22673

Sample legitimate URLs:
['https://manua.ls', 'https://strato.de', 'https://techmeme.com', 'https://mahabhumi.gov.in', 'https://goldenfrog.com', 'https://interactivedns.com', 'https://brocardi.it', 'https://www.northwell.edu', 'https://pitch.com', 'https://www.post.at']


In [175]:
import random

random.seed(42)

# ---------------------------------------------
# Generate realistic legitimate URL structures
# ---------------------------------------------

path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example"
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2"
]

augmented_urls = set()

# Use 5,000 additional examples
target_count = 5000

while len(augmented_urls) < target_count:

    domain = random.choice(legitimate_domains)

    # 70% path URLs
    # 30% query URLs
    if random.random() < 0.70:
        suffix = random.choice(path_templates)
    else:
        suffix = random.choice(query_templates)

    new_url = "https://" + domain + suffix

    augmented_urls.add(new_url)

augmented_urls = list(augmented_urls)

print("Number of augmented legitimate URLs:", len(augmented_urls))

print("\nSample augmented URLs:")
for url in augmented_urls[:20]:
    print(url)

Number of augmented legitimate URLs: 5000

Sample augmented URLs:
https://holley.com/questions?page=2
https://anime-planet.com/search/results
https://hotstar.com/about
https://savethechildren.org.uk/products
https://aau.at/search?q=python
https://sogetel.net/about-us
https://xn--69-lqix0ea5a7kua2b8lc3gd.com/search?query=test
https://www.renttherunway.com/products/item
https://xnxx.place/help
https://ecma-international.org/search/results
https://excellmedia.net/wiki/example
https://boreus.de/blog
https://verdi.de/contact
https://plaud.ai/products/item
https://ut.ee/products/item
https://www.vectorstock.com/help
https://benesse.ne.jp/search?query=test
https://top.pl/help
https://clarodigital.net/products
https://tedra.es/questions?page=2


In [176]:
import pandas as pd

aug_df = pd.DataFrame({
    "url": augmented_urls,
    "label": "legitimate"
})

# ---------------------------------------------
# Check path/query distribution
# ---------------------------------------------

aug_df["has_path"] = aug_df["url"].apply(
    lambda x: "/" in x.split("://", 1)[1]
)

aug_df["has_query"] = aug_df["url"].apply(
    lambda x: "?" in x
)

aug_df["url_length"] = aug_df["url"].str.len()

aug_df["path_length"] = aug_df["url"].apply(
    lambda x: len(urlparse(x).path)
)

aug_df["query_length"] = aug_df["url"].apply(
    lambda x: len(urlparse(x).query)
)

print("Augmented dataset shape:", aug_df.shape)

print("\nPath distribution:")
print(aug_df["has_path"].value_counts())

print("\nQuery distribution:")
print(aug_df["has_query"].value_counts())

print("\nAverage URL length:")
print(round(aug_df["url_length"].mean(), 2))

print("\nAverage path length:")
print(round(aug_df["path_length"].mean(), 2))

print("\nAverage query length:")
print(round(aug_df["query_length"].mean(), 2))

print("\nExamples with queries:")
print(
    aug_df[aug_df["has_query"]]["url"].head(10).to_list()
)

Augmented dataset shape: (5000, 7)

Path distribution:
has_path
True    5000
Name: count, dtype: int64

Query distribution:
has_query
False    3567
True     1433
Name: count, dtype: int64

Average URL length:
32.92

Average path length:
8.59

Average query length:
3.16

Examples with queries:
['https://holley.com/questions?page=2', 'https://aau.at/search?q=python', 'https://xn--69-lqix0ea5a7kua2b8lc3gd.com/search?query=test', 'https://benesse.ne.jp/search?query=test', 'https://tedra.es/questions?page=2', 'https://forabank.ru/search?page=2', 'https://formstack.com/search?q=python', 'https://rambler-co.ru/article?id=12345', 'https://guitarplayer.com/questions?page=2', 'https://narvar.com/search?page=2']


In [177]:
print(repr(augmented_urls[0]))

print("\nFirst 5 raw URLs:")
for url in augmented_urls[:5]:
    print(repr(url))

'https://holley.com/questions?page=2'

First 5 raw URLs:
'https://holley.com/questions?page=2'
'https://anime-planet.com/search/results'
'https://hotstar.com/about'
'https://savethechildren.org.uk/products'
'https://aau.at/search?q=python'


In [178]:
import re

def clean_augmented_url(url):
    # If URL is stored as Markdown:
    # [https://example.com/path](https://example.com/path)
    match = re.match(r"^\[.*?\]\((https?://.*?)\)$", url)

    if match:
        return match.group(1)

    return url


# Clean all augmented URLs
augmented_urls = [
    clean_augmented_url(url)
    for url in augmented_urls
]

# Remove duplicates after cleaning
augmented_urls = list(set(augmented_urls))

print("Number of cleaned URLs:", len(augmented_urls))

print("\nFirst 5 cleaned URLs:")
for url in augmented_urls[:5]:
    print(repr(url))

Number of cleaned URLs: 5000

First 5 cleaned URLs:
'https://holley.com/questions?page=2'
'https://anime-planet.com/search/results'
'https://hotstar.com/about'
'https://savethechildren.org.uk/products'
'https://aau.at/search?q=python'


In [179]:
def clean_augmented_url(url):
    if "](" in url and url.endswith(")"):
        url = url.split("](", 1)[1]
        url = url[:-1]  # remove final )

    return url


augmented_urls = [
    clean_augmented_url(url)
    for url in augmented_urls
]

# Remove duplicates
augmented_urls = list(set(augmented_urls))

print("Number of cleaned URLs:", len(augmented_urls))

print("\nFirst 5 cleaned URLs:")
for url in augmented_urls[:5]:
    print(repr(url))

Number of cleaned URLs: 5000

First 5 cleaned URLs:
'https://holley.com/questions?page=2'
'https://anime-planet.com/search/results'
'https://hotstar.com/about'
'https://savethechildren.org.uk/products'
'https://aau.at/search?q=python'


In [180]:
url = augmented_urls[0]

print("RAW:")
print(url)

print("\nREPR:")
print(repr(url))

print("\nTYPE:")
print(type(url))

print("\nContains '](':", "](" in url)
print("Ends with ')':", url.endswith(")"))

print("\nCharacters around '](':")
pos = url.find("](")
print(pos)

if pos != -1:
    print(repr(url[pos:pos+20]))

RAW:
https://holley.com/questions?page=2

REPR:
'https://holley.com/questions?page=2'

TYPE:
<class 'str'>

Contains '](': False
Ends with ')': False

Characters around '](':
-1


In [181]:
url = augmented_urls[0]

print("Character-by-character:")
for i, char in enumerate(url):
    print(i, repr(char), ord(char))

Character-by-character:
0 'h' 104
1 't' 116
2 't' 116
3 'p' 112
4 's' 115
5 ':' 58
6 '/' 47
7 '/' 47
8 'h' 104
9 'o' 111
10 'l' 108
11 'l' 108
12 'e' 101
13 'y' 121
14 '.' 46
15 'c' 99
16 'o' 111
17 'm' 109
18 '/' 47
19 'q' 113
20 'u' 117
21 'e' 101
22 's' 115
23 't' 116
24 'i' 105
25 'o' 111
26 'n' 110
27 's' 115
28 '?' 63
29 'p' 112
30 'a' 97
31 'g' 103
32 'e' 101
33 '=' 61
34 '2' 50


In [182]:
# Extract the same 11 features used by the final XGBoost model

augmented_features = pd.DataFrame(
    [extract_url_features(url) for url in augmented_urls]
)

augmented_features = augmented_features[url_features_final]

print("Augmented feature shape:", augmented_features.shape)

print("\nAverage feature values:")
display(augmented_features.mean().round(4))

print("\nFeature ranges:")
display(
    pd.DataFrame({
        "min": augmented_features.min(),
        "max": augmented_features.max()
    }).round(4)
)

Augmented feature shape: (5000, 11)

Average feature values:


,0
entropy,4.0056
url_length,32.9188
num_www,0.1590
ratio_digits_url,0.0200
num_subdomains,0.1586
length_hostname,12.8798
num_dots,1.2336
num_hyphens,0.0956
ratio_digits_host,0.0096
num_question_mark,0.2866



Feature ranges:


,min,max
entropy,3.1158,4.9094
url_length,17.0000,68.0000
num_www,0.0000,1.0000
ratio_digits_url,0.0000,0.2500
num_subdomains,0.0000,1.0000
length_hostname,4.0000,32.0000
num_dots,1.0000,4.0000
num_hyphens,0.0000,3.0000
ratio_digits_host,0.0000,0.6250
num_question_mark,0.0000,1.0000


In [183]:
# Get original training data
train_data = df_clean.iloc[train_idx].copy()

# Extract URL features for the original training URLs
original_train_features = pd.DataFrame(
    [extract_url_features(url) for url in train_data["url"]]
)

original_train_features["label"] = train_data["label"].values

# Separate legitimate and phishing
original_legitimate_features = original_train_features[
    original_train_features["label"] == "legitimate"
].drop(columns="label")

original_phishing_features = original_train_features[
    original_train_features["label"] == "phishing"
].drop(columns="label")


# ---------------------------------------------
# Compare means
# ---------------------------------------------

comparison = pd.DataFrame({
    "Original Legitimate": original_legitimate_features.mean(),
    "Augmented Legitimate": augmented_features.mean(),
    "Original Phishing": original_phishing_features.mean()
})

display(comparison.round(4))

,Original Legitimate,Augmented Legitimate,Original Phishing
entropy,3.6999,4.0056,4.4399
url_length,20.8210,32.9188,68.4806
num_www,0.1598,0.1590,0.0293
ratio_digits_url,0.0056,0.0200,0.0827
num_subdomains,0.1594,0.1586,0.7701
length_hostname,12.8210,12.8798,20.3169
num_dots,1.2368,1.2336,2.0331
num_hyphens,0.0564,0.0956,0.8524
ratio_digits_host,0.0098,0.0096,0.0546
num_question_mark,0.0000,0.2866,0.2283


In [184]:
# Use the existing training split
train_data = df_clean.iloc[train_idx].copy()

# Check which labels each domain has
domain_label_check = (
    train_data
    .groupby("domain")["label"]
    .unique()
)

# Keep only domains whose training label is exclusively legitimate
pure_legitimate_domains = domain_label_check[
    domain_label_check.apply(
        lambda labels: len(labels) == 1 and labels[0] == "legitimate"
    )
].index.tolist()

print("Total training domains:",
      train_data["domain"].nunique())

print("Pure legitimate domains:",
      len(pure_legitimate_domains))

print("Ambiguous/mixed domains:",
      train_data["domain"].nunique() - len(pure_legitimate_domains))

print("\nSample pure legitimate domains:")
print(pure_legitimate_domains[:20])

Total training domains: 32723
Pure legitimate domains: 20330
Ambiguous/mixed domains: 12393

Sample pure legitimate domains:
['007names.net', '009.am', '00web.net', '0123tt.ru', '01net.com', '021.rs', '0aa0cf0637d66c0d.com', '1-grid.co.uk', '1-hostingservice.com', '1-ofd.ru', '1000logos.net', '10010.com', '10086.cn', '100lw.com', '100sp.ru', '101domain.com', '101hotels.com', '103.by', '10fastfingers.com', '10jqka.com.cn']


In [185]:
import random

random.seed(42)

# ---------------------------------------------
# Legitimate URL structures
# ---------------------------------------------

path_templates = [
    "/search",
    "/search/results",
    "/products",
    "/products/item",
    "/about",
    "/about-us",
    "/contact",
    "/help",
    "/support",
    "/blog",
    "/news",
    "/article",
    "/category",
    "/login",
    "/account",
    "/profile",
    "/settings",
    "/questions/12345",
    "/wiki/example"
]

query_templates = [
    "/search?q=python",
    "/search?q=machine+learning",
    "/search?query=test",
    "/search?page=2",
    "/products?id=12345",
    "/products?category=electronics",
    "/article?id=12345",
    "/questions?page=2",
    "/search?q=laptop&page=2"
]


# ---------------------------------------------
# Generate augmentation
# ---------------------------------------------

augmented_urls = set()

target_count = 5000

while len(augmented_urls) < target_count:

    domain = random.choice(pure_legitimate_domains)

    # 70% path-only
    # 30% query-containing
    if random.random() < 0.70:
        suffix = random.choice(path_templates)
    else:
        suffix = random.choice(query_templates)

    augmented_urls.add("https://" + domain + suffix)


augmented_urls = list(augmented_urls)

print("Augmented legitimate URLs:", len(augmented_urls))

print("\nFirst 10 URLs:")
for url in augmented_urls[:10]:
    print(url)

Augmented legitimate URLs: 5000

First 10 URLs:
https://g2288.com/search?q=python
https://buddy4study.com/category
https://scandinavianhosting.se/products?id=12345
https://ufrgs.br/products?id=12345
https://xhamsterflirt.com/questions?page=2
https://neworleans.com/products?id=12345
https://onlinetestpad.com/about
https://zalando-prive.it/login
https://portalinmobiliario.com/questions?page=2
https://getvim.com/about


In [186]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# --------------------------------------------------
# 1. Prepare original training data
# --------------------------------------------------

X_train_final = X_train[url_features_final].copy()

# --------------------------------------------------
# 2. Extract features from augmented URLs
# --------------------------------------------------

augmented_features = pd.DataFrame(
    [extract_url_features(url) for url in augmented_urls]
)

augmented_features = augmented_features[url_features_final]

# All augmented URLs are legitimate
augmented_labels = np.zeros(len(augmented_features), dtype=int)


# --------------------------------------------------
# 3. Combine original + augmented training data
# --------------------------------------------------

X_train_augmented = pd.concat(
    [
        X_train_final.reset_index(drop=True),
        augmented_features.reset_index(drop=True)
    ],
    ignore_index=True
)

y_train_augmented = np.concatenate(
    [
        y_train,
        augmented_labels
    ]
)

print("Original training samples:", len(X_train_final))
print("Augmented samples:", len(augmented_features))
print("Final training samples:", len(X_train_augmented))


# --------------------------------------------------
# 4. Train CANDIDATE XGBoost model
# --------------------------------------------------

augmented_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

augmented_xgb_model.fit(
    X_train_augmented,
    y_train_augmented
)


# --------------------------------------------------
# 5. Evaluate on the ORIGINAL untouched test set
# --------------------------------------------------

X_test_final = X_test[url_features_final].copy()

current_pred = final_xgb_model.predict(X_test_final)
current_prob = final_xgb_model.predict_proba(X_test_final)[:, 1]

augmented_pred = augmented_xgb_model.predict(X_test_final)
augmented_prob = augmented_xgb_model.predict_proba(X_test_final)[:, 1]


# --------------------------------------------------
# 6. Compare both models
# --------------------------------------------------

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current Model": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "Augmented Model": [
        accuracy_score(y_test, augmented_pred),
        precision_score(y_test, augmented_pred),
        recall_score(y_test, augmented_pred),
        f1_score(y_test, augmented_pred),
        roc_auc_score(y_test, augmented_prob)
    ]
})

display(comparison.round(6))

Original training samples: 47323
Augmented samples: 5000
Final training samples: 52323


,Metric,Current Model,Augmented Model
0,Accuracy,0.993115,0.973661
1,Precision,0.998813,0.998001
2,Recall,0.988541,0.953577
3,F1,0.993650,0.975284
4,ROC-AUC,0.997955,0.997234


In [187]:
validation_urls = [
    # Legitimate URLs with paths / queries
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://github.com",
    "https://github.com/explore",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://stackoverflow.com/questions/12345",

    # Suspicious-looking URLs
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "http://secure-login-account-verification-12345.com/login"
]

validation_results = []

for url in validation_urls:

    features = extract_url_features(url)

    X_input = pd.DataFrame(
        [features],
        columns=url_features_final
    )

    current_probability = final_xgb_model.predict_proba(
        X_input
    )[0][1]

    augmented_probability = augmented_xgb_model.predict_proba(
        X_input
    )[0][1]

    validation_results.append({
        "URL": url,
        "Current Model %": round(current_probability * 100, 2),
        "Augmented Model %": round(augmented_probability * 100, 2)
    })

validation_results_df = pd.DataFrame(validation_results)

display(validation_results_df)

,URL,Current Model %,Augmented Model %
0,https://www.google.com,0.550000,0.620000
1,https://www.google.com/search,99.230003,69.449997
2,https://www.google.com/search?q=python,99.980003,98.379997
3,https://github.com,0.480000,0.510000
4,https://github.com/explore,99.790001,25.410000
5,https://en.wikipedia.org/wiki/Machine_learning,100.000000,99.989998
6,https://stackoverflow.com/questions/12345,100.000000,9.640000
7,http://paypal-account-security-check-48291.com...,88.290001,75.199997
8,http://free-prize-winner-92731.com/claim,82.680000,9.320000
9,http://secure-login-account-verification-12345...,93.720001,87.470001


In [188]:
# Create a much smaller augmentation set
# from our already generated 5,000 examples.

random.seed(42)

small_augmented_urls = random.sample(
    augmented_urls,
    500
)

print("Small augmentation size:", len(small_augmented_urls))

print("\nSample URLs:")
for url in small_augmented_urls[:10]:
    print(url)

Small augmentation size: 500

Sample URLs:
https://arvig.net/products?category=electronics
https://rechargeapps.com/support
https://sankuai.com/support
https://relativity.one/help
https://voxlink.ru/about
https://ayads.co/settings
https://im.cz/account
https://hdb.gov.sg/search?query=test
https://bnovo.ru/settings
https://orderly.network/profile


In [189]:
# --------------------------------------------------
# Prepare the 500 augmented examples
# --------------------------------------------------

small_augmented_features = pd.DataFrame(
    [extract_url_features(url) for url in small_augmented_urls]
)

small_augmented_features = small_augmented_features[
    url_features_final
]

# All are intended as legitimate examples
small_augmented_labels = np.zeros(
    len(small_augmented_features),
    dtype=int
)


# --------------------------------------------------
# Combine with ORIGINAL training data
# --------------------------------------------------

X_train_small_aug = pd.concat(
    [
        X_train[url_features_final].reset_index(drop=True),
        small_augmented_features.reset_index(drop=True)
    ],
    ignore_index=True
)

y_train_small_aug = np.concatenate(
    [
        y_train,
        small_augmented_labels
    ]
)

print("Original training samples:", len(X_train))
print("Small augmentation:", len(small_augmented_features))
print("Final training samples:", len(X_train_small_aug))


# --------------------------------------------------
# Train candidate model
# --------------------------------------------------

small_augmented_xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

small_augmented_xgb_model.fit(
    X_train_small_aug,
    y_train_small_aug
)

print("\n500-example augmented model trained successfully.")

Original training samples: 47323
Small augmentation: 500
Final training samples: 47823

500-example augmented model trained successfully.


In [190]:
# --------------------------------------------------
# Evaluate current model vs 500-example model
# on the SAME untouched test set
# --------------------------------------------------

X_test_final = X_test[url_features_final].copy()

# Current baseline
current_pred = final_xgb_model.predict(X_test_final)
current_prob = final_xgb_model.predict_proba(X_test_final)[:, 1]

# 500-example augmented model
small_aug_pred = small_augmented_xgb_model.predict(X_test_final)
small_aug_prob = small_augmented_xgb_model.predict_proba(X_test_final)[:, 1]


# --------------------------------------------------
# Comparison
# --------------------------------------------------

comparison_500 = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current Model": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "500-Augmented Model": [
        accuracy_score(y_test, small_aug_pred),
        precision_score(y_test, small_aug_pred),
        recall_score(y_test, small_aug_pred),
        f1_score(y_test, small_aug_pred),
        roc_auc_score(y_test, small_aug_prob)
    ]
})

display(comparison_500.round(6))

,Metric,Current Model,500-Augmented Model
0,Accuracy,0.993115,0.989833
1,Precision,0.998813,0.998507
2,Recall,0.988541,0.982812
3,F1,0.993650,0.990597
4,ROC-AUC,0.997955,0.997908


In [191]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):

    predictions = (current_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions)
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(4))

,Threshold,Accuracy,Precision,Recall,F1
0,0.10,0.9857,0.9813,0.9928,0.9870
1,0.15,0.9906,0.9908,0.9919,0.9913
2,0.20,0.9924,0.9945,0.9915,0.9930
3,0.25,0.9933,0.9962,0.9915,0.9938
4,0.30,0.9933,0.9970,0.9906,0.9938
5,0.35,0.9934,0.9979,0.9900,0.9940
6,0.40,0.9937,0.9987,0.9897,0.9942
7,0.45,0.9934,0.9987,0.9891,0.9939
8,0.50,0.9931,0.9988,0.9885,0.9937
9,0.55,0.9930,0.9993,0.9878,0.9935


In [192]:
comparison_urls = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://www.github.com",
    "https://www.github.com/explore"
]

feature_comparison = pd.DataFrame(
    [
        extract_url_features(url)
        for url in comparison_urls
    ],
    index=comparison_urls
)

display(feature_comparison[url_features_final].round(4))

,entropy,url_length,num_www,ratio_digits_url,num_subdomains,length_hostname,num_dots,num_hyphens,ratio_digits_host,num_question_mark,num_eq
https://www.google.com,3.6635,22,1,0.0,1,14,2,0,0.0,0,0
https://www.google.com/search,3.8833,29,1,0.0,1,14,2,0,0.0,0,0
https://www.google.com/search?q=python,4.2211,38,1,0.0,1,14,2,0,0.0,1,1
https://www.github.com,3.7544,22,1,0.0,1,14,2,0,0.0,0,0
https://www.github.com/explore,4.0981,30,1,0.0,1,14,2,0,0.0,0,0


In [193]:
comparison_urls = [
    "https://www.google.com",
    "https://www.google.com/search",
    "https://www.google.com/search?q=python",
    "https://www.github.com",
    "https://www.github.com/explore"
]

# Extract features
comparison_features = pd.DataFrame(
    [
        extract_url_features(url)
        for url in comparison_urls
    ]
)[url_features_final]

# Calculate SHAP values
comparison_shap = shap_explainer.shap_values(comparison_features)

# Display SHAP contribution for each URL
for i, url in enumerate(comparison_urls):

    shap_df = pd.DataFrame({
        "Feature": url_features_final,
        "Feature Value": comparison_features.iloc[i].values,
        "SHAP Contribution": comparison_shap[i]
    })

    shap_df["Absolute SHAP"] = shap_df["SHAP Contribution"].abs()

    shap_df = shap_df.sort_values(
        "Absolute SHAP",
        ascending=False
    )

    print("\n" + "=" * 80)
    print(url)
    print("=" * 80)

    display(
        shap_df[
            ["Feature", "Feature Value", "SHAP Contribution"]
        ].head(6).round(4)
    )


https://www.google.com


,Feature,Feature Value,SHAP Contribution
1,url_length,22.0000,-3.6139
2,num_www,1.0000,-2.3671
4,num_subdomains,1.0000,1.0842
0,entropy,3.6635,-0.6997
5,length_hostname,14.0000,0.6853
3,ratio_digits_url,0.0000,-0.2808



https://www.google.com/search


,Feature,Feature Value,SHAP Contribution
5,length_hostname,14.0000,4.4622
1,url_length,29.0000,2.9921
2,num_www,1.0000,-2.6592
4,num_subdomains,1.0000,0.8632
0,entropy,3.8833,-0.4588
3,ratio_digits_url,0.0000,-0.2679



https://www.google.com/search?q=python


,Feature,Feature Value,SHAP Contribution
1,url_length,38.0000,5.4972
5,length_hostname,14.0000,3.5089
2,num_www,1.0000,-2.0065
4,num_subdomains,1.0000,0.9419
0,entropy,4.2211,0.7883
3,ratio_digits_url,0.0000,-0.2809



https://www.github.com


,Feature,Feature Value,SHAP Contribution
1,url_length,22.0000,-3.5975
2,num_www,1.0000,-2.5035
4,num_subdomains,1.0000,0.9993
5,length_hostname,14.0000,0.6805
0,entropy,3.7544,-0.6234
3,ratio_digits_url,0.0000,-0.2856



https://www.github.com/explore


,Feature,Feature Value,SHAP Contribution
5,length_hostname,14.0000,4.0358
2,num_www,1.0000,-2.6787
1,url_length,30.0000,2.6378
4,num_subdomains,1.0000,0.9994
0,entropy,4.0981,0.4781
3,ratio_digits_url,0.0000,-0.4044


In [194]:
# Calculate SHAP interaction values
interaction_values = shap_explainer.shap_interaction_values(
    comparison_features
)

important_features = [
    "url_length",
    "length_hostname",
    "entropy",
    "num_www"
]

for i, url in enumerate(comparison_urls):

    print("\n" + "=" * 80)
    print(url)
    print("=" * 80)

    interaction_matrix = pd.DataFrame(
        interaction_values[i],
        index=url_features_final,
        columns=url_features_final
    )

    # Show interactions among the important features
    display(
        interaction_matrix.loc[
            important_features,
            important_features
        ].round(4)
    )


https://www.google.com


,url_length,length_hostname,entropy,num_www
url_length,-1.5417,-2.2955,0.3151,-0.1793
length_hostname,-2.2955,2.8346,0.0255,0.1299
entropy,0.3151,0.0255,-1.3454,0.1992
num_www,-0.1793,0.1299,0.1992,-1.3135



https://www.google.com/search


,url_length,length_hostname,entropy,num_www
url_length,1.4450,1.4166,0.4778,-0.3403
length_hostname,1.4166,2.6760,0.1175,0.3673
entropy,0.4778,0.1175,-1.0881,-0.0026
num_www,-0.3403,0.3673,-0.0026,-1.3739



https://www.google.com/search?q=python


,url_length,length_hostname,entropy,num_www
url_length,5.3560,0.4914,-0.2531,0.0778
length_hostname,0.4914,2.6697,-0.0727,0.4535
entropy,-0.2531,-0.0727,0.9350,0.0904
num_www,0.0778,0.4535,0.0904,-1.6375



https://www.github.com


,url_length,length_hostname,entropy,num_www
url_length,-1.5241,-2.3047,0.3406,-0.1900
length_hostname,-2.3047,2.8450,0.0242,0.1234
entropy,0.3406,0.0242,-1.0707,0.0864
num_www,-0.1900,0.1234,0.0864,-1.2625



https://www.github.com/explore


,url_length,length_hostname,entropy,num_www
url_length,2.1223,1.1262,-0.3184,-0.2402
length_hostname,1.1262,2.8812,-0.1434,0.3189
entropy,-0.3184,-0.1434,1.1296,-0.1099
num_www,-0.2402,0.3189,-0.1099,-1.2931


In [195]:
# --------------------------------------------------
# Remove url_length for an ablation experiment
# --------------------------------------------------

url_features_no_length = [
    feature for feature in url_features_final
    if feature != "url_length"
]

print("Features being used:")
print(url_features_no_length)

print("\nNumber of features:", len(url_features_no_length))


# --------------------------------------------------
# Prepare training and test data
# --------------------------------------------------

X_train_no_length_xgb = X_train[url_features_no_length].copy()
X_test_no_length_xgb = X_test[url_features_no_length].copy()


# --------------------------------------------------
# Train experimental XGBoost
# --------------------------------------------------

xgb_no_length = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_no_length.fit(
    X_train_no_length_xgb,
    y_train
)


# --------------------------------------------------
# Evaluate
# --------------------------------------------------

no_length_pred = xgb_no_length.predict(
    X_test_no_length_xgb
)

no_length_prob = xgb_no_length.predict_proba(
    X_test_no_length_xgb
)[:, 1]


no_length_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current Model": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "Without url_length": [
        accuracy_score(y_test, no_length_pred),
        precision_score(y_test, no_length_pred),
        recall_score(y_test, no_length_pred),
        f1_score(y_test, no_length_pred),
        roc_auc_score(y_test, no_length_prob)
    ]
})

display(no_length_results.round(6))

Features being used:
['entropy', 'num_www', 'ratio_digits_url', 'num_subdomains', 'length_hostname', 'num_dots', 'num_hyphens', 'ratio_digits_host', 'num_question_mark', 'num_eq']

Number of features: 10


,Metric,Current Model,Without url_length
0,Accuracy,0.993115,0.976703
1,Precision,0.998813,0.993936
2,Recall,0.988541,0.963126
3,F1,0.993650,0.978288
4,ROC-AUC,0.997955,0.993981


In [196]:
# --------------------------------------------------
# XGBoost experiment: shallower trees
# --------------------------------------------------

xgb_depth3 = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_depth3.fit(
    X_train[url_features_final],
    y_train
)


# --------------------------------------------------
# Evaluate on the same test set
# --------------------------------------------------

depth3_pred = xgb_depth3.predict(
    X_test[url_features_final]
)

depth3_prob = xgb_depth3.predict_proba(
    X_test[url_features_final]
)[:, 1]


depth3_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current depth=6": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "depth=3": [
        accuracy_score(y_test, depth3_pred),
        precision_score(y_test, depth3_pred),
        recall_score(y_test, depth3_pred),
        f1_score(y_test, depth3_pred),
        roc_auc_score(y_test, depth3_prob)
    ]
})

display(depth3_results.round(6))

,Metric,Current depth=6,depth=3
0,Accuracy,0.993115,0.987831
1,Precision,0.998813,0.998054
2,Recall,0.988541,0.979580
3,F1,0.993650,0.988731
4,ROC-AUC,0.997955,0.997358


In [197]:
xgb_depth5 = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_depth5.fit(
    X_train[url_features_final],
    y_train
)

depth5_pred = xgb_depth5.predict(
    X_test[url_features_final]
)

depth5_prob = xgb_depth5.predict_proba(
    X_test[url_features_final]
)[:, 1]

depth5_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Current depth=6": [
        accuracy_score(y_test, current_pred),
        precision_score(y_test, current_pred),
        recall_score(y_test, current_pred),
        f1_score(y_test, current_pred),
        roc_auc_score(y_test, current_prob)
    ],

    "depth=5": [
        accuracy_score(y_test, depth5_pred),
        precision_score(y_test, depth5_pred),
        recall_score(y_test, depth5_pred),
        f1_score(y_test, depth5_pred),
        roc_auc_score(y_test, depth5_prob)
    ]
})

display(depth5_results.round(6))

,Metric,Current depth=6,depth=5
0,Accuracy,0.993115,0.992234
1,Precision,0.998813,0.999107
2,Recall,0.988541,0.986631
3,F1,0.993650,0.992830
4,ROC-AUC,0.997955,0.997969


In [198]:
validation_results_depth = []

for url in validation_urls:

    features = extract_url_features(url)

    X_input = pd.DataFrame(
        [features],
        columns=url_features_final
    )

    depth6_probability = final_xgb_model.predict_proba(
        X_input
    )[0][1]

    depth5_probability = xgb_depth5.predict_proba(
        X_input
    )[0][1]

    validation_results_depth.append({
        "URL": url,
        "Depth 6 %": round(depth6_probability * 100, 2),
        "Depth 5 %": round(depth5_probability * 100, 2)
    })

depth_comparison_df = pd.DataFrame(
    validation_results_depth
)

display(depth_comparison_df)

,URL,Depth 6 %,Depth 5 %
0,https://www.google.com,0.550000,0.550000
1,https://www.google.com/search,99.230003,99.080002
2,https://www.google.com/search?q=python,99.980003,99.980003
3,https://github.com,0.480000,0.450000
4,https://github.com/explore,99.790001,99.720001
5,https://en.wikipedia.org/wiki/Machine_learning,100.000000,100.000000
6,https://stackoverflow.com/questions/12345,100.000000,100.000000
7,http://paypal-account-security-check-48291.com...,88.290001,78.379997
8,http://free-prize-winner-92731.com/claim,82.680000,72.930000
9,http://secure-login-account-verification-12345...,93.720001,89.110001


In [199]:
test_urls_complete = [
    # Legitimate websites
    "https://www.google.com",
    "https://www.google.com/search?q=python",
    "https://github.com/explore",
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://stackoverflow.com/questions/12345",

    # Suspicious-looking URLs
    "http://paypal-account-security-check-48291.com/verify",
    "http://free-prize-winner-92731.com/claim",
    "http://secure-login-account-verification-12345.com/login"
]

complete_results = []

for url in test_urls_complete:

    print("\n" + "=" * 90)
    print("Testing:", url)
    print("=" * 90)

    try:
        result = analyze_url_complete(url)

        complete_results.append(result)

        print("Prediction:", result["prediction"])
        print("Phishing Probability:", result["phishing_probability"], "%")
        print("Legitimate Probability:", result["legitimate_probability"], "%")
        print("Risk Level:", result["risk_level"])

        print("\nLive Domain Intelligence:")
        print("Domain Age:", result["domain_age_days"], "days")
        print("DNS Resolvable:", result["dns_resolvable"])
        print("Number of IPs:", result["num_ips"])
        print("TTL:", result["ttl"])
        print("Name Servers:", result["num_name_servers"])

        print("\nPhishing Reasons:")
        for reason in result["phishing_reasons"]:
            print("-", reason)

        print("\nLegitimate Reasons:")
        for reason in result["legitimate_reasons"]:
            print("-", reason)

    except Exception as e:
        print("ERROR:", e)


Testing: https://www.google.com
Prediction: Legitimate
Phishing Probability: 0.55 %
Legitimate Probability: 99.45 %
Risk Level: Low

Live Domain Intelligence:
Domain Age: 10606 days
DNS Resolvable: Yes
Number of IPs: 8
TTL: 300
Name Servers: 4

Phishing Reasons:
- Number of subdomains: 1 (contributed toward the phishing prediction)
- Hostname length: 14 (contributed toward the phishing prediction)
- Digit ratio in the hostname: 0.00% (contributed toward the phishing prediction)

Legitimate Reasons:
- URL length: 22 characters (contributed toward the legitimate prediction)
- Number of 'www' occurrences: 1 (contributed toward the legitimate prediction)
- URL character entropy: 3.66 (contributed toward the legitimate prediction)

Testing: https://www.google.com/search?q=python
Prediction: Phishing
Phishing Probability: 99.98 %
Legitimate Probability: 0.02 %
Risk Level: High

Live Domain Intelligence:
Domain Age: 10606 days
DNS Resolvable: Yes
Number of IPs: 8
TTL: 300
Name Servers: 4

Ph

In [200]:
def format_live_value(value, suffix=""):
    if value is None or value == -1:
        return "Unavailable"
    return f"{value}{suffix}"


def analyze_url_complete(url):
    result = analyze_url(url)

    phishing_probability = float(result["phishing_probability"])

    if phishing_probability >= 0.80:
        risk_level = "High"
    elif phishing_probability >= 0.50:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    feature_explanation = generate_readable_feature_explanation(url)

    return {
        "url": url,
        "prediction": result["prediction"],

        "phishing_probability": round(
            float(result["phishing_probability"]) * 100, 2
        ),

        "legitimate_probability": round(
            float(result["legitimate_probability"]) * 100, 2
        ),

        "risk_level": risk_level,

        "phishing_reasons": feature_explanation["phishing_reasons"],
        "legitimate_reasons": feature_explanation["legitimate_reasons"],

        # Better handling of unavailable live data
        "domain_age_days": format_live_value(
            result["domain_age"], " days"
        ),

        "dns_resolvable": (
            "Yes"
            if result["dns_resolvable"] == 1
            else "No"
        ),

        "num_ips": format_live_value(
            result["num_ips"]
        ),

        "ttl": format_live_value(
            result["ttl"]
        ),

        "num_name_servers": format_live_value(
            result["num_name_servers"]
        )
    }


print("Updated analyze_url_complete() successfully.")

Updated analyze_url_complete() successfully.


In [201]:
test_result = analyze_url_complete(
    "http://free-prize-winner-92731.com/claim"
)

print(test_result)

{'url': 'http://free-prize-winner-92731.com/claim', 'prediction': 'Phishing', 'phishing_probability': 82.68, 'legitimate_probability': 17.32, 'risk_level': 'High', 'phishing_reasons': ['URL length: 40 characters (contributed toward the phishing prediction)', 'URL character entropy: 4.40 (contributed toward the phishing prediction)', 'Digit ratio in the URL: 12.50% (contributed toward the phishing prediction)'], 'legitimate_reasons': ['Digit ratio in the hostname: 18.52% (contributed toward the legitimate prediction)', 'Hostname length: 27 (contributed toward the legitimate prediction)', 'Number of dots in the URL: 1 (contributed toward the legitimate prediction)'], 'domain_age_days': 'Unavailable', 'dns_resolvable': 'No', 'num_ips': '0', 'ttl': 'Unavailable', 'num_name_servers': '0'}


In [202]:
import joblib
import json

# Save final trained model
joblib.dump(final_xgb_model, "phishing_xgb_model.pkl")

# Save feature names
joblib.dump(url_features_final, "feature_names.pkl")

# Save feature descriptions
joblib.dump(feature_descriptions, "feature_descriptions.pkl")

print("Files saved successfully:")
print("- phishing_xgb_model.pkl")
print("- feature_names.pkl")
print("- feature_descriptions.pkl")

Files saved successfully:
- phishing_xgb_model.pkl
- feature_names.pkl
- feature_descriptions.pkl


In [203]:
from google.colab import files

files.download("phishing_xgb_model.pkl")
files.download("feature_names.pkl")
files.download("feature_descriptions.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>